# DermaMatch AI — Recommendation Engine
## Final, clean, execution-ordered notebook

This is **Part 2** of the DermaMatch AI project.

It consumes the processed `recommendation_catalog.csv` created by the Part 1 data pipeline and builds the complete recommendation layer:

**Processed Catalog → Query Understanding → Sentence-Transformer Embeddings → ChromaDB Retrieval → Hard Constraints → Ingredient Intelligence → Review/Preference/Rating Signals → Diversity-aware Ranking → Deterministic Explanation → `recommend()`**

### Design guarantees

- Every required function is defined **before** it is used.
- `catalog_by_id` is created immediately after the catalog is loaded.
- Ingredient scoring and avoided-ingredient filtering share the same canonical ingredient profile.
- Missing ingredient data is represented honestly.
- The ranking formula and implementation use the same six documented weights.
- The notebook contains deterministic diagnostics and final assertions.
- The production Flask layer can call `recommend()` without duplicating ranking logic.

# Notebook Roadmap

1. Introduction  
2. Environment & imports  
3. Locate and load processed catalog  
4. Catalog validation  
5. Build product documents  
6. Load Sentence Transformer  
7. Generate product embeddings  
8. Create ChromaDB collection  
9. Index products  
10. Build query normalizer  
11. Build semantic retrieval  
12. Build hard constraint filtering  
13. Build Ingredient Intelligence Engine  
14. Ingredient root-cause diagnostic  
15. Build review scoring  
16. Build preference scoring  
17. Build rating-quality scoring  
18. Build diversity-aware ranking  
19. Build deterministic explanations  
20. Build final `recommend()` function  
21. Recommendation test suite  
22. Ingredient sanity tests  
23. Edge-case tests  
24. Ranking sensitivity analysis  
25. Save application artifacts  
26. Final validation  
27. Handoff to Flask + Streamlit

# 2. Environment & Imports

Run this notebook in the same Python environment in which you intend to run the final project.

The installation cell is intentionally minimal. For deployment, the exact versions should later be pinned in `requirements.txt` and Docker rather than relying on notebook-time installation.

In [6]:
%pip install -q -U sentence-transformers chromadb

Note: you may need to restart the kernel to use updated packages.


In [7]:
from __future__ import annotations

import ast
import json
import math
import os
import re
import sys
import time
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

import chromadb
from sentence_transformers import SentenceTransformer

print("ChromaDB:", chromadb.__version__)
print("Sentence Transformers:", __import__("sentence_transformers").__version__)
print("Environment imports: PASS")

Python: 3.11.15
Pandas: 2.2.3
NumPy: 2.4.6
ChromaDB: 1.5.9
Sentence Transformers: 6.0.0
Environment imports: PASS


# 3. Locate and Load the Processed Catalog

The notebook accepts the processed catalog in any of these normal project locations:

- `data/processed/recommendation_catalog.csv`
- `data/recommendation_catalog.csv`
- `recommendation_catalog.csv`

It also searches below the current working directory for the exact filename.

No fabricated fallback dataset is used. If the file is missing, the notebook stops with a clear `FileNotFoundError`.

In [8]:
# ============================================================
# 3. Locate and Load the Processed Catalog
# ============================================================

from pathlib import Path

# This notebook is inside:
# ORBO.ai/notebooks/
#
# Therefore the project root is one directory above the notebook.
NOTEBOOK_DIR = Path.cwd()

# Find the ORBO.ai project root robustly.
PROJECT_ROOT = None

candidate = NOTEBOOK_DIR.resolve()

# Check current directory and its parents for the expected project structure.
for folder in [candidate, *candidate.parents]:
    processed_catalog = (
        folder
        / "data"
        / "processed"
        / "recommendation_catalog.csv"
    )

    if processed_catalog.exists():
        PROJECT_ROOT = folder
        CATALOG_PATH = processed_catalog
        break

# Fallback for the standard notebook location:
if PROJECT_ROOT is None:
    standard_root = NOTEBOOK_DIR.parent.resolve()
    standard_catalog = (
        standard_root
        / "data"
        / "processed"
        / "recommendation_catalog.csv"
    )

    if standard_catalog.exists():
        PROJECT_ROOT = standard_root
        CATALOG_PATH = standard_catalog

# Final check
if PROJECT_ROOT is None or not CATALOG_PATH.exists():
    raise FileNotFoundError(
        "\nCould not find recommendation_catalog.csv.\n\n"
        "Expected project structure:\n"
        "ORBO.ai/\n"
        "├── data/\n"
        "│   └── processed/\n"
        "│       └── recommendation_catalog.csv\n"
        "└── notebooks/\n"
        "    └── 02_DermaMatch_Recommendation_Engine_FINAL_NO_ERROR.ipynb\n\n"
        f"Current working directory: {Path.cwd()}\n"
    )

print("=" * 80)
print("CATALOG LOCATION")
print("=" * 80)
print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Catalog path:", CATALOG_PATH)
print("Catalog exists:", CATALOG_PATH.exists())
print("Catalog size (MB):", round(CATALOG_PATH.stat().st_size / (1024**2), 2))
print("Catalog location validation: PASS")

CATALOG LOCATION
Current working directory: d:\CODE\ORBO.ai\notebooks
Project root: D:\CODE\ORBO.ai
Catalog path: D:\CODE\ORBO.ai\data\processed\recommendation_catalog.csv
Catalog exists: True
Catalog size (MB): 13.35
Catalog location validation: PASS


In [9]:
catalog = pd.read_csv(CATALOG_PATH, low_memory=False)

print("Catalog loaded successfully.")
print("Rows:", len(catalog))
print("Columns:", len(catalog.columns))
print("Shape:", catalog.shape)

display(catalog.head(5))

Catalog loaded successfully.
Rows: 2420
Columns: 69
Shape: (2420, 69)


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,theme_drying_share,theme_fragrance_share,theme_irritation_share,theme_breakout_share,theme_absorption_share,theme_sticky_share,theme_texture_share,theme_effective_share,skin_type_profile,recommendation_document
0,P439055,GENIUS Sleeping Collagen Moisturizer,6018,Algenist,33910,4.5413,1321.0,2 oz/ 60 mL,Size,2 oz/ 60 mL,...,0.273278,0.241484,0.129447,0.137774,0.077971,0.024981,0.604845,0.216503,"oily: 0.08, dry: 0.21, combination: 0.56, norm...",Product: GENIUS Sleeping Collagen Moisturizer ...
1,P421277,GENIUS Liquid Collagen Serum,6018,Algenist,67870,4.0259,1159.0,1 oz / 30 mL,Size,1 oz / 30 mL,...,0.086281,0.184642,0.086281,0.097498,0.069025,0.062985,0.358067,0.271786,"oily: 0.08, dry: 0.20, combination: 0.50, norm...",Product: GENIUS Liquid Collagen Serum | Brand:...
2,P467602,Triple Algae Eye Renewal Balm Eye Cream,6018,Algenist,17890,4.5306,1142.0,NaN,NaN,NaN,...,0.050788,0.096322,0.119089,0.008757,0.092820,0.024518,0.395797,0.341506,"oily: 0.11, dry: 0.18, combination: 0.56, norm...",Product: Triple Algae Eye Renewal Balm Eye Cre...
3,P432045,GENIUS Liquid Collagen Lip Treatment,6018,Algenist,44448,3.8721,649.0,.5 oz / 15 mL,Size,.5 oz / 15 mL,...,0.232666,0.023112,0.077042,0.006163,0.057011,0.038521,0.309707,0.382126,"oily: 0.10, dry: 0.23, combination: 0.48, norm...",Product: GENIUS Liquid Collagen Lip Treatment ...
4,P311143,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,6018,Algenist,27278,4.4134,508.0,1 oz,Size,1 oz,...,0.145669,0.177165,0.145669,0.151575,0.135827,0.045276,0.303150,0.190945,"oily: 0.09, dry: 0.12, combination: 0.43, norm...",Product: SUBLIME DEFENSE Ultra Lightweight UV ...


In [10]:
required_columns = [
    "product_id",
    "product_name",
    "brand_name",
]

missing_required = [
    col for col in required_columns
    if col not in catalog.columns
]

if missing_required:
    raise KeyError(
        "Missing required catalog columns: "
        + ", ".join(missing_required)
    )

catalog["product_id"] = catalog["product_id"].astype(str)

# CRITICAL: create the lookup BEFORE any function that depends on it.
catalog_by_id = catalog.set_index(
    "product_id",
    drop=False,
)

print("Canonical product lookup created.")
print("Lookup rows:", len(catalog_by_id))
print("Unique product IDs:", catalog_by_id.index.nunique())

assert catalog["product_id"].notna().all()
assert catalog["product_id"].is_unique
assert catalog_by_id.index.is_unique

print("Catalog key validation: PASS")

Canonical product lookup created.
Lookup rows: 2420
Unique product IDs: 2420
Catalog key validation: PASS


# 4. Catalog Validation

This section reports the fields that will drive the recommendation system.

The actual column list comes from the processed Part 1 catalog; optional fields are detected rather than assumed.

In [11]:
def columns_containing(*terms: str) -> List[str]:
    return [
        c for c in catalog.columns
        if any(term.lower() in c.lower() for term in terms)
    ]


ingredient_columns = columns_containing("ingredient")
skin_columns = [
    c for c in catalog.columns
    if c.lower().startswith("skin_share_")
]
theme_columns = [
    c for c in catalog.columns
    if c.lower().startswith("theme_")
]
price_columns = columns_containing("price")
review_columns = columns_containing("review", "rating", "recommendation")

print("Ingredient columns:", ingredient_columns)
print("Skin-share columns:", skin_columns)
print("Theme columns:", theme_columns)
print("Price columns:", price_columns)
print("Review/rating columns:", review_columns)

Ingredient columns: ['ingredients', 'ingredients_clean', 'ingredient_tokens', 'ingredient_count']
Skin-share columns: ['skin_share_dry', 'skin_share_unknown', 'skin_share_combination', 'skin_share_normal', 'skin_share_oily', 'skin_share_sensitive']
Theme columns: ['theme_lightweight_count', 'theme_greasy_count', 'theme_hydrating_count', 'theme_drying_count', 'theme_fragrance_count', 'theme_irritation_count', 'theme_breakout_count', 'theme_absorption_count', 'theme_sticky_count', 'theme_texture_count', 'theme_effective_count', 'theme_lightweight_share', 'theme_greasy_share', 'theme_hydrating_share', 'theme_drying_share', 'theme_fragrance_share', 'theme_irritation_share', 'theme_breakout_share', 'theme_absorption_share', 'theme_sticky_share', 'theme_texture_share', 'theme_effective_share']
Price columns: ['price_usd', 'value_price_usd', 'sale_price_usd', 'child_max_price', 'child_min_price', 'effective_price_usd']
Review/rating columns: ['rating', 'reviews', 'review_count_observed', 'rev

In [12]:
print("Missing-value summary for core fields:")
core_fields = [
    c for c in [
        "product_id",
        "product_name",
        "brand_name",
        "ingredients",
        "ingredients_clean",
        "ingredient_tokens",
        "effective_price_usd",
        "rating",
        "review_avg_rating",
        "review_count_observed",
        "recommendation_rate",
    ]
    if c in catalog.columns
]

summary_rows = []
for col in core_fields:
    summary_rows.append({
        "field": col,
        "missing": int(catalog[col].isna().sum()),
        "missing_pct": round(float(catalog[col].isna().mean() * 100), 2),
        "unique": int(catalog[col].nunique(dropna=True)),
    })

display(pd.DataFrame(summary_rows))

Missing-value summary for core fields:


,field,missing,missing_pct,unique
0,product_id,0,0.00,2420
1,product_name,0,0.00,2401
2,brand_name,0,0.00,143
3,ingredients,134,5.54,2101
4,ingredients_clean,134,5.54,2098
5,ingredient_tokens,0,0.00,2069
6,effective_price_usd,0,0.00,226
7,rating,69,2.85,1672
8,review_avg_rating,69,2.85,1779
9,review_count_observed,0,0.00,876


# 5. Build Product Documents

The semantic document contains descriptive information that should influence embedding similarity.

Strict numeric constraints such as budget are kept out of the semantic score and are handled explicitly later.

In [13]:
def safe_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    text = str(value).strip()
    return "" if text.lower() in {"nan", "none", "null"} else text


def normalize_token(value: Any) -> str:
    text = safe_text(value).lower()
    text = re.sub(r"[^a-z0-9\s%_-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_product_document(row: pd.Series) -> str:
    parts = []

    field_labels = [
        ("product_name", "Product"),
        ("brand_name", "Brand"),
        ("primary_category", "Primary category"),
        ("secondary_category", "Secondary category"),
        ("tertiary_category", "Product type"),
        ("highlights", "Highlights"),
        ("ingredients_clean", "Ingredients"),
        ("ingredients", "Ingredients"),
        ("skin_type_profile", "Skin-type profile"),
    ]

    seen_labels = set()

    for field, label in field_labels:
        if field not in row.index:
            continue

        value = safe_text(row[field])
        if not value:
            continue

        if label == "Ingredients":
            # Do not duplicate cleaned + raw ingredients.
            if "Ingredients" in seen_labels:
                continue

        parts.append(f"{label}: {value}")
        seen_labels.add(label)

    theme_names = []
    for col in theme_columns:
        value = pd.to_numeric(row.get(col), errors="coerce")
        if pd.notna(value) and float(value) > 0:
            name = re.sub(r"^theme_", "", col.lower())
            name = re.sub(r"_share$|_count$", "", name)
            theme_names.append(name.replace("_", " "))

    if theme_names:
        parts.append("Review themes: " + ", ".join(sorted(set(theme_names))))

    return " | ".join(parts)


catalog["product_document"] = catalog.apply(
    build_product_document,
    axis=1,
)

empty_document_count = int(
    catalog["product_document"].str.strip().eq("").sum()
)

print("Product documents:", len(catalog))
print("Empty documents:", empty_document_count)

assert empty_document_count == 0
print("Product-document validation: PASS")

print("\nExample product document:")
print(catalog.loc[0, "product_document"][:1200])

Product documents: 2420
Empty documents: 0
Product-document validation: PASS

Example product document:
Product: GENIUS Sleeping Collagen Moisturizer | Brand: Algenist | Primary category: Skincare | Secondary category: Moisturizers | Product type: Moisturizers | Highlights: ['Vegan', 'Good for: Loss of firmness', 'Collagen', 'Hypoallergenic', 'Without Parabens', 'Best for Dry, Combo, Normal Skin'] | Ingredients: Collagen (Vegan)*, Water (Aqua, Eau), Ethylhexyl Palmitate, Oryza Sativa (Rice) Bran Extract, Caprylic/Capric Triglyceride, Glycerin, Cetearyl Methicone, Dimethicone, Cetearyl Alcohol, Pyrus Malus (Apple) Fruit Extract, Chlorella Protothecoides Oil, Polysorbate 60, Glyceryl Glucoside , Polyglyceryl-2 Stearate, Parachlorella Beijerinckii Exopolysaccharides, Collagen Amino Acids (Vegan)*, Ceramide NP, Silybum Marianum Fruit Extract, Helianthus Annuus (Sunflower) Extract, Rosmarinus Officinalis (Rosemary) Leaf Extract, Tocopherol, Carnosine, Sodium Stearoyl Glutamate, Caprylyl Gly

# 6. Load Sentence Transformer

In [14]:
EMBEDDING_MODEL_NAME = os.getenv(
    "DERMAMATCH_EMBEDDING_MODEL",
    "BAAI/bge-small-en-v1.5",
)

print("Loading model:", EMBEDDING_MODEL_NAME)

embedder = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
)

_probe = embedder.encode(
    ["DermaMatch skincare recommendation"],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

EMBEDDING_DIM = int(_probe.shape[-1])

print("Embedding dimension:", EMBEDDING_DIM)
print("Embedding model loaded: PASS")

Loading model: BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimension: 384
Embedding model loaded: PASS


# 7. Generate Product Embeddings

In [15]:
ARTIFACT_DIR = PROJECT_ROOT / "data" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_PATH = ARTIFACT_DIR / "product_embeddings.npy"
EMBEDDING_IDS_PATH = ARTIFACT_DIR / "product_embedding_ids.json"

product_documents = catalog["product_document"].tolist()
product_ids = catalog["product_id"].astype(str).tolist()

print("Documents:", len(product_documents))
print("Batch size: 64")

start_time = time.perf_counter()

product_embeddings = embedder.encode(
    product_documents,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

product_embeddings = np.asarray(
    product_embeddings,
    dtype=np.float32,
)

elapsed = time.perf_counter() - start_time

print("\nEmbedding generation complete.")
print("Matrix shape:", product_embeddings.shape)
print("Embedding dimension:", product_embeddings.shape[1])
print("Time (seconds):", round(elapsed, 2))
print(
    "Mean vector norm:",
    round(
        float(
            np.linalg.norm(
                product_embeddings,
                axis=1,
            ).mean()
        ),
        6,
    ),
)

assert product_embeddings.shape == (
    len(catalog),
    EMBEDDING_DIM,
)
assert np.isfinite(product_embeddings).all()

np.save(
    EMBEDDINGS_PATH,
    product_embeddings,
)
EMBEDDING_IDS_PATH.write_text(
    json.dumps(product_ids, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Embedding validation: PASS")

Documents: 2420
Batch size: 64


Batches:   0%|          | 0/38 [00:00<?, ?it/s]


Embedding generation complete.
Matrix shape: (2420, 384)
Embedding dimension: 384
Time (seconds): 35.35
Mean vector norm: 1.0
Embedding validation: PASS


# 8. Create ChromaDB Collection

In [16]:
CHROMA_PATH = ARTIFACT_DIR / "chroma"
CHROMA_PATH.mkdir(parents=True, exist_ok=True)

CHROMA_COLLECTION_NAME = "dermamatch_products"

print("ChromaDB path:", CHROMA_PATH)
print("Collection:", CHROMA_COLLECTION_NAME)

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH),
)

try:
    chroma_client.delete_collection(
        CHROMA_COLLECTION_NAME,
    )
    print("Existing collection removed.")
except Exception:
    print("No previous collection found.")

collection = chroma_client.get_or_create_collection(
    name=CHROMA_COLLECTION_NAME,
    metadata={
        "description": "DermaMatch skincare product embeddings",
        "embedding_model": EMBEDDING_MODEL_NAME,
        "hnsw:space": "cosine",
    },
)

print("Collection count:", collection.count())

ChromaDB path: D:\CODE\ORBO.ai\data\artifacts\chroma
Collection: dermamatch_products
Existing collection removed.
Collection count: 0


# 9. Store Product Vectors + Metadata

In [17]:
def numeric_value(
    row: pd.Series,
    candidates: Sequence[str],
    default: float = 0.0,
) -> float:
    for col in candidates:
        if col not in row.index:
            continue
        value = pd.to_numeric(
            row[col],
            errors="coerce",
        )
        if pd.notna(value):
            return float(value)
    return float(default)


def make_metadata(row: pd.Series) -> Dict[str, Any]:
    metadata = {
        "product_id": str(row["product_id"]),
        "product_name": safe_text(row.get("product_name")),
        "brand_name": safe_text(row.get("brand_name")),
        "primary_category": safe_text(row.get("primary_category")),
        "secondary_category": safe_text(row.get("secondary_category")),
        "tertiary_category": safe_text(row.get("tertiary_category")),
        "effective_price_usd": numeric_value(
            row,
            ["effective_price_usd", "sale_price_usd", "price_usd"],
            default=0.0,
        ),
        "rating": numeric_value(
            row,
            ["rating"],
            default=0.0,
        ),
        "review_count": numeric_value(
            row,
            ["review_count_observed", "reviews"],
            default=0.0,
        ),
        "recommendation_rate": numeric_value(
            row,
            ["recommendation_rate"],
            default=0.0,
        ),
    }

    return metadata


chroma_ids = [
    f"product_{pid}"
    for pid in product_ids
]

chroma_documents = catalog[
    "product_document"
].tolist()

chroma_metadatas = [
    make_metadata(row)
    for _, row in catalog.iterrows()
]

BATCH_SIZE = 500

for start in range(
    0,
    len(chroma_ids),
    BATCH_SIZE,
):
    end = min(
        start + BATCH_SIZE,
        len(chroma_ids),
    )

    collection.upsert(
        ids=chroma_ids[start:end],
        embeddings=product_embeddings[start:end].tolist(),
        documents=chroma_documents[start:end],
        metadatas=chroma_metadatas[start:end],
    )

    print(
        f"Indexed {end:,}/{len(chroma_ids):,}"
    )

print("Final ChromaDB count:", collection.count())

assert collection.count() == len(catalog)
print("ChromaDB indexing validation: PASS")

Indexed 500/2,420
Indexed 1,000/2,420
Indexed 1,500/2,420
Indexed 2,000/2,420
Indexed 2,420/2,420
Final ChromaDB count: 2420
ChromaDB indexing validation: PASS


# 10. Query Normalizer + Ingredient Vocabulary

The query normalizer converts free text and structured quiz inputs into one stable internal representation.

In [18]:
SKIN_TYPE_ALIASES = {
    "oily": ["oily"],
    "dry": ["dry", "dehydrated"],
    "combination": ["combination", "combo"],
    "normal": ["normal"],
    "sensitive": ["sensitive"],
}

CONCERN_TERMS = {
    "acne": ["acne", "breakout", "break out", "pimple", "blemish"],
    "hydration": ["hydration", "hydrating", "hydrate", "dryness"],
    "dark_spots": [
        "dark spot",
        "dark spots",
        "hyperpigmentation",
        "pigmentation",
    ],
    "anti_aging": [
        "anti aging",
        "anti-aging",
        "wrinkle",
        "fine line",
        "firming",
    ],
    "oil_control": [
        "oil control",
        "oily",
        "shine control",
    ],
}

PREFERENCE_TERMS = {
    "lightweight": [
        "lightweight",
        "light weight",
        "light texture",
    ],
    "fragrance_free": [
        "fragrance free",
        "fragrance-free",
        "no fragrance",
        "unscented",
    ],
    "non_comedogenic": [
        "non-comedogenic",
        "noncomedogenic",
        "won't clog pores",
        "does not clog pores",
    ],
    "non_greasy": [
        "non-greasy",
        "non greasy",
        "not greasy",
        "greasy-free",
    ],
}

CATEGORY_ALIASES = {
    "cleanser": ["cleanser", "cleanse", "face wash", "facial wash"],
    "moisturizer": [
        "moisturizer",
        "moisturiser",
        "moisturizing cream",
        "face cream",
    ],
    "sunscreen": ["sunscreen", "sun screen", "spf"],
    "serum": ["serum"],
    "mask": ["mask", "face mask"],
    "eye care": ["eye cream", "eye care", "eye treatment"],
    "treatment": ["treatment", "acne treatment"],
}

INGREDIENT_ALIASES = {
    "salicylic acid": [
        "salicylic acid",
        "bha",
        "beta hydroxy acid",
    ],
    "niacinamide": [
        "niacinamide",
        "nicotinamide",
    ],
    "zinc": [
        "zinc",
        "zinc oxide",
        "zinc gluconate",
        "zinc pca",
    ],
    "hyaluronic acid": [
        "hyaluronic acid",
        "sodium hyaluronate",
        "hydrolyzed hyaluronic acid",
        "hydrolysed hyaluronic acid",
    ],
    "vitamin c": [
        "vitamin c",
        "ascorbic acid",
        "l ascorbic acid",
        "ascorbic",
    ],
    "azelaic acid": ["azelaic acid"],
    "benzoyl peroxide": ["benzoyl peroxide"],
    "retinol": ["retinol"],
    "retinal": ["retinal", "retinaldehyde"],
    "peptide": ["peptide", "peptides"],
    "ceramide": ["ceramide", "ceramides"],
    "glycerin": ["glycerin", "glycerine", "glycerol"],
    "squalane": ["squalane"],
    "fragrance": ["fragrance", "parfum", "perfume"],
    "alcohol denat": [
        "alcohol denat",
        "denatured alcohol",
        "sd alcohol",
        "sd alcohol 40",
    ],
}

CONCERN_INGREDIENT_WEIGHTS = {
    "acne": {
        "salicylic acid": 1.00,
        "benzoyl peroxide": 1.00,
        "azelaic acid": 0.90,
        "niacinamide": 0.65,
    },
    "oil_control": {
        "niacinamide": 1.00,
        "salicylic acid": 0.90,
        "zinc": 0.70,
    },
    "hydration": {
        "hyaluronic acid": 1.00,
        "glycerin": 0.95,
        "squalane": 0.80,
        "ceramide": 0.80,
    },
    "dark_spots": {
        "vitamin c": 1.00,
        "niacinamide": 0.85,
        "azelaic acid": 0.80,
    },
    "anti_aging": {
        "retinol": 1.00,
        "retinal": 0.95,
        "peptide": 0.75,
        "vitamin c": 0.70,
    },
}

ALIAS_TO_CANONICAL = {
    normalize_token(alias): canonical
    for canonical, aliases in INGREDIENT_ALIASES.items()
    for alias in aliases
}


def detect_first_match(
    text: str,
    alias_map: Dict[str, Sequence[str]],
) -> Optional[str]:
    normalized = normalize_token(text)

    for canonical, aliases in alias_map.items():
        for alias in aliases:
            if re.search(
                rf"(?<![a-z0-9]){re.escape(normalize_token(alias))}(?![a-z0-9])",
                normalized,
            ):
                return canonical

    return None


def detect_all_matches(
    text: str,
    alias_map: Dict[str, Sequence[str]],
) -> List[str]:
    normalized = normalize_token(text)
    found = []

    for canonical, aliases in alias_map.items():
        if any(
            re.search(
                rf"(?<![a-z0-9]){re.escape(normalize_token(alias))}(?![a-z0-9])",
                normalized,
            )
            for alias in aliases
        ):
            found.append(canonical)

    return found


def detect_requested_ingredients(
    text: str,
) -> List[str]:
    normalized = normalize_token(text)
    if not normalized:
        return []

    found = []

    for alias, canonical in sorted(
        ALIAS_TO_CANONICAL.items(),
        key=lambda x: len(x[0]),
        reverse=True,
    ):
        if re.search(
            rf"(?<![a-z0-9]){re.escape(alias)}(?![a-z0-9])",
            normalized,
        ):
            found.append(canonical)

    return list(dict.fromkeys(found))


def extract_budget(
    text: str,
) -> Optional[float]:
    normalized = normalize_token(text)

    patterns = [
        r"(?:under|below|less than|up to|upto)\s*\$?\s*(\d+(?:\.\d+)?)",
        r"\$\s*(\d+(?:\.\d+)?)",
        r"(?:budget|price)\s*(?:of)?\s*\$?\s*(\d+(?:\.\d+)?)",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            normalized,
        )
        if match:
            return float(match.group(1))

    return None


def normalize_query(
    query: Optional[str] = None,
    skin_type: Optional[str] = None,
    concerns: Optional[Sequence[str]] = None,
    category: Optional[str] = None,
    budget_max: Optional[float] = None,
    preferred_terms: Optional[Sequence[str]] = None,
    avoid_ingredients: Optional[Sequence[str]] = None,
) -> Dict[str, Any]:
    query_text = safe_text(query)

    detected_concerns = (
        detect_all_matches(
            query_text,
            CONCERN_TERMS,
        )
        if query_text
        else []
    )

    detected_preferences = (
        detect_all_matches(
            query_text,
            PREFERENCE_TERMS,
        )
        if query_text
        else []
    )

    detected_skin_type = (
        detect_first_match(
            query_text,
            SKIN_TYPE_ALIASES,
        )
        if query_text
        else None
    )

    detected_category = (
        detect_first_match(
            query_text,
            CATEGORY_ALIASES,
        )
        if query_text
        else None
    )

    detected_ingredients = (
        detect_requested_ingredients(
            query_text
        )
        if query_text
        else []
    )

    detected_budget = (
        extract_budget(query_text)
        if query_text
        else None
    )

    return {
        "query_text": query_text,
        "skin_type": normalize_token(skin_type) if skin_type else None,
        "concerns": [
            normalize_token(x)
            for x in (concerns or [])
            if safe_text(x)
        ],
        "category": normalize_token(category) if category else None,
        "budget_max": (
            float(budget_max)
            if budget_max is not None
            else None
        ),
        "preferred_terms": [
            normalize_token(x)
            for x in (preferred_terms or [])
            if safe_text(x)
        ],
        "avoid_ingredients": [
            normalize_token(x)
            for x in (avoid_ingredients or [])
            if safe_text(x)
        ],
        "detected_skin_type": detected_skin_type,
        "detected_concerns": detected_concerns,
        "detected_preferences": detected_preferences,
        "detected_category": detected_category,
        "detected_ingredients": detected_ingredients,
        "detected_budget_max": detected_budget,
        "effective_skin_type": (
            normalize_token(skin_type)
            if skin_type
            else detected_skin_type
        ),
        "effective_concerns": list(dict.fromkeys(
            [
                *[
                    normalize_token(x)
                    for x in (concerns or [])
                    if safe_text(x)
                ],
                *detected_concerns,
            ]
        )),
        "effective_category": (
            normalize_token(category)
            if category
            else detected_category
        ),
        "effective_preferences": list(dict.fromkeys(
            [
                *[
                    normalize_token(x)
                    for x in (preferred_terms or [])
                    if safe_text(x)
                ],
                *detected_preferences,
            ]
        )),
        "effective_ingredients": detected_ingredients,
        "effective_budget_max": (
            float(budget_max)
            if budget_max is not None
            else detected_budget
        ),
    }


query_example = normalize_query(
    "lightweight sunscreen for oily acne-prone skin with zinc oxide under $30"
)

print(
    json.dumps(
        query_example,
        indent=2,
    )
)

{
  "query_text": "lightweight sunscreen for oily acne-prone skin with zinc oxide under $30",
  "skin_type": null,
  "concerns": [],
  "category": null,
  "budget_max": null,
  "preferred_terms": [],
  "avoid_ingredients": [],
  "detected_skin_type": "oily",
  "detected_concerns": [
    "acne",
    "oil_control"
  ],
  "detected_preferences": [
    "lightweight"
  ],
  "detected_category": "sunscreen",
  "detected_ingredients": [
    "zinc"
  ],
  "detected_budget_max": 30.0,
  "effective_skin_type": "oily",
  "effective_concerns": [
    "acne",
    "oil_control"
  ],
  "effective_category": "sunscreen",
  "effective_preferences": [
    "lightweight"
  ],
  "effective_ingredients": [
    "zinc"
  ],
  "effective_budget_max": 30.0
}


# 11. Semantic Retrieval

The query is embedded using the same model used for products.

Default workflow:

**Top 50 retrieved → hard constraints → score → diversity-aware Top 5**

In [19]:
def build_query_text(q: Dict[str, Any]) -> str:
    parts = []

    if q.get("query_text"):
        parts.append(q["query_text"])

    if q.get("effective_skin_type"):
        parts.append(
            f"skin type: {q['effective_skin_type']}"
        )

    if q.get("effective_concerns"):
        parts.append(
            "concerns: "
            + ", ".join(q["effective_concerns"])
        )

    if q.get("effective_category"):
        parts.append(
            f"category: {q['effective_category']}"
        )

    if q.get("effective_preferences"):
        parts.append(
            "preferences: "
            + ", ".join(q["effective_preferences"])
        )

    if q.get("effective_ingredients"):
        parts.append(
            "ingredients: "
            + ", ".join(q["effective_ingredients"])
        )

    return " | ".join(
        part for part in parts if part
    )


def semantic_retrieve(
    q: Dict[str, Any],
    candidate_k: int = 50,
) -> pd.DataFrame:
    query_text = build_query_text(q)

    if not query_text:
        raise ValueError(
            "A query or structured recommendation signal is required."
        )

    query_vector = embedder.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )[0]

    result = collection.query(
        query_embeddings=[
            query_vector.tolist()
        ],
        n_results=min(
            int(candidate_k),
            len(catalog),
        ),
        include=[
            "documents",
            "metadatas",
            "distances",
        ],
    )

    ids = result.get("ids", [[]])[0]
    distances = result.get(
        "distances",
        [[]],
    )[0]
    metadatas = result.get(
        "metadatas",
        [[]],
    )[0]

    rows = []

    for chroma_id, distance, metadata in zip(
        ids,
        distances,
        metadatas,
    ):
        row = dict(metadata)
        row["chroma_id"] = chroma_id
        row["cosine_distance"] = float(distance)
        row["semantic_similarity"] = float(
            np.clip(
                1.0 - float(distance),
                0.0,
                1.0,
            )
        )
        rows.append(row)

    retrieved = pd.DataFrame(rows)

    if retrieved.empty:
        raise RuntimeError(
            "ChromaDB returned zero candidates."
        )

    return retrieved


retrieval_test = semantic_retrieve(
    query_example,
    candidate_k=10,
)

print(
    "Retrieved candidates:",
    len(retrieval_test),
)

display(
    retrieval_test[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "effective_price_usd",
            "semantic_similarity",
        ]
    ]
)

Retrieved candidates: 10


,product_id,product_name,brand_name,primary_category,effective_price_usd,semantic_similarity
0,P483658,Mineral Sunscreen Zinc Oxide Broad Spectrum SP...,First Aid Beauty,Skincare,28.0,0.814903
1,P449188,Green Defense Daily Mineral Sunscreen SPF 30,Farmacy,Skincare,36.0,0.788423
2,P446909,Zincscreen 100% Mineral Sunscreen Lotion SPF 4...,Supergoop!,Skincare,44.0,0.782111
3,P467976,(Re)setting 100% Mineral Powder Sunscreen SPF ...,Supergoop!,Skincare,35.0,0.780471
4,P482258,Mineral Face Sunscreen Sheer Matte SPF 30,COOLA,Skincare,36.0,0.780093
5,P482321,Mini Mineral Sheerscreen Sunscreen SPF 30 PA+++,Supergoop!,Skincare,22.0,0.776047
6,P482540,Correct & Protect Broad Spectrum SPF 45| PA++++,Murad,Skincare,69.0,0.775329
7,P470057,Mineral Sheerscreen Sunscreen SPF 30 PA+++,Supergoop!,Skincare,38.0,0.774348
8,P456410,Squalane + Zinc Sheer Mineral Sunscreen SPF 30...,Biossance,Skincare,34.0,0.767352
9,P419221,Umbra Tinte Physical Daily Defense SPF 30,Drunk Elephant,Skincare,36.0,0.767311


# 12. Hard Constraint Filtering

Budget, requested category, and explicitly avoided ingredients are handled outside the embedding model.

This prevents a semantically similar product from violating a user's strict constraint.

In [20]:
def parse_avoid_terms(
    q: Dict[str, Any],
) -> List[str]:
    return [
        normalize_token(x)
        for x in q.get("avoid_ingredients", [])
        if safe_text(x)
    ]


def apply_hard_filters(
    retrieved: pd.DataFrame,
    q: Dict[str, Any],
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    if retrieved.empty:
        return (
            retrieved.copy(),
            {
                "input": 0,
                "after_filter": 0,
            },
        )

    out = retrieved.copy()
    stats = {"input": len(out)}

    budget = q.get(
        "effective_budget_max"
    )

    if budget is not None and "effective_price_usd" in out.columns:
        price = pd.to_numeric(
            out["effective_price_usd"],
            errors="coerce",
        )
        out = out[
            price.notna()
            & (price <= float(budget))
        ].copy()

    stats["after_budget"] = len(out)

    category = normalize_token(
        q.get("effective_category")
    )

    if category:
        cols = [
            c for c in [
                "primary_category",
                "secondary_category",
                "tertiary_category",
            ]
            if c in out.columns
        ]

        if cols:
            category_mask = pd.Series(
                False,
                index=out.index,
            )

            pattern = re.escape(category)

            for col in cols:
                category_mask |= (
                    out[col]
                    .fillna("")
                    .map(normalize_token)
                    .str.contains(
                        pattern,
                        regex=True,
                    )
                )

            out = out[category_mask].copy()

    stats["after_category"] = len(out)

    # Ingredient avoidance uses the same canonical profile as scoring.
    avoid_terms = parse_avoid_terms(q)
    rejected = 0

    if avoid_terms:
        keep = []

        for pid in out["product_id"].astype(str):
            profile = ingredient_profiles.get(
                pid,
                get_product_ingredient_profile(pid),
            )

            searchable = set(
                profile["canonical_ingredients"]
            )
            searchable.update(
                profile["raw_tokens"]
            )
            searchable_text = normalize_token(
                profile["raw_text"]
                + " "
                + " ".join(searchable)
            )

            violation = False

            for term in avoid_terms:
                canonical = canonicalize_ingredient(
                    term
                )

                if canonical and canonical in searchable:
                    violation = True
                    break

                aliases = INGREDIENT_ALIASES.get(
                    canonical,
                    [term],
                )

                if any(
                    normalize_token(alias)
                    in searchable_text
                    for alias in aliases
                    if normalize_token(alias)
                ):
                    violation = True
                    break

            keep.append(not violation)
            rejected += int(violation)

        out = out[
            np.asarray(
                keep,
                dtype=bool,
            )
        ].copy()

    stats["rejected_by_avoided_ingredient"] = rejected
    stats["after_avoid_ingredients"] = len(out)
    stats["after_filter"] = len(out)

    return (
        out.reset_index(drop=True),
        stats,
    )

# 13. Ingredient Intelligence Engine

This is the final deterministic ingredient layer.

### It supports

- exact/canonical matching
- alias matching
- concern → ingredient relevance
- explicit ingredient requests
- ingredient coverage
- hard avoidance
- honest missing-data handling

It uses **one canonical product ingredient profile** everywhere.

In [21]:
# One canonical ingredient profile is used by scoring,
# avoided-ingredient filtering, explanations and diagnostics.

def parse_ingredient_tokens(
    value: Any,
) -> List[str]:
    text = safe_text(value)

    if not text:
        return []

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return [
                    normalize_token(x)
                    for x in parsed
                    if safe_text(x)
                ]
        except Exception:
            pass

        try:
            parsed = ast.literal_eval(text)
            if isinstance(
                parsed,
                (list, tuple, set),
            ):
                return [
                    normalize_token(x)
                    for x in parsed
                    if safe_text(x)
                ]
        except Exception:
            pass

    return [
        normalize_token(x)
        for x in re.split(
            r"[,;|\n]+",
            text,
        )
        if normalize_token(x)
    ]


def canonicalize_ingredient(
    value: Any,
) -> Optional[str]:
    phrase = normalize_token(value)

    if not phrase:
        return None

    if phrase in ALIAS_TO_CANONICAL:
        return ALIAS_TO_CANONICAL[phrase]

    # Remove concentration values such as "20%" and trailing numbers.
    cleaned = re.sub(
        r"\b\d+(?:\.\d+)?%?\b",
        " ",
        phrase,
    )
    cleaned = re.sub(
        r"\s+",
        " ",
        cleaned,
    ).strip()

    if cleaned in ALIAS_TO_CANONICAL:
        return ALIAS_TO_CANONICAL[cleaned]

    # Longest alias wins.
    for alias, canonical in sorted(
        ALIAS_TO_CANONICAL.items(),
        key=lambda x: len(x[0]),
        reverse=True,
    ):
        if phrase.startswith(alias + " "):
            return canonical

    return None


def get_product_ingredient_profile(
    product_id: str,
) -> Dict[str, Any]:
    pid = str(product_id)

    if pid not in catalog_by_id.index:
        return {
            "available": False,
            "ingredient_data_available": False,
            "raw_text": "",
            "normalized_text": "",
            "raw_tokens": [],
            "canonical_ingredients": [],
            "ingredient_count": 0,
        }

    row = catalog_by_id.loc[pid]

    clean_text = safe_text(
        row.get("ingredients_clean", "")
    )
    raw_text = safe_text(
        row.get("ingredients", "")
    )

    ingredient_text = (
        clean_text
        if clean_text
        else raw_text
    )

    serialized_tokens = parse_ingredient_tokens(
        row.get("ingredient_tokens", "")
    )

    parsed_tokens = [
        normalize_token(part)
        for part in re.split(
            r",|;|\||\n",
            ingredient_text,
        )
        if normalize_token(part)
    ]

    raw_tokens = list(
        dict.fromkeys(
            serialized_tokens
            + parsed_tokens
        )
    )

    canonical = set()

    # Canonicalize complete ingredient phrases.
    for token in raw_tokens:
        canonical_name = canonicalize_ingredient(
            token
        )
        if canonical_name:
            canonical.add(
                canonical_name
            )

    # Search the complete ingredient text for multi-word aliases.
    normalized_text = normalize_token(
        ingredient_text
    )

    for alias, canonical_name in sorted(
        ALIAS_TO_CANONICAL.items(),
        key=lambda x: len(x[0]),
        reverse=True,
    ):
        if re.search(
            rf"(?<![a-z0-9]){re.escape(alias)}(?![a-z0-9])",
            normalized_text,
        ):
            canonical.add(
                canonical_name
            )

    available = bool(
        ingredient_text.strip()
    ) or bool(raw_tokens)

    return {
        "available": bool(available),
        "ingredient_data_available": bool(available),
        "raw_text": ingredient_text,
        "normalized_text": normalized_text,
        "raw_tokens": sorted(set(raw_tokens)),
        "canonical_ingredients": sorted(canonical),
        "ingredient_count": len(canonical),
    }


def requested_ingredient_weights(
    q: Dict[str, Any],
) -> Dict[str, float]:
    weights: Dict[str, float] = {}

    # Concern-driven ingredients.
    for concern in q.get(
        "effective_concerns",
        [],
    ):
        concern_key = normalize_token(
            concern
        )

        for ingredient, weight in CONCERN_INGREDIENT_WEIGHTS.get(
            concern_key,
            {},
        ).items():
            weights[ingredient] = max(
                weights.get(ingredient, 0.0),
                float(weight),
            )

    # Explicit ingredient requests always receive full relevance.
    explicit = set(
        q.get(
            "effective_ingredients",
            [],
        )
    )

    explicit.update(
        detect_requested_ingredients(
            q.get("query_text", "")
        )
    )

    for ingredient in explicit:
        canonical = canonicalize_ingredient(
            ingredient
        )
        if canonical:
            weights[canonical] = max(
                weights.get(canonical, 0.0),
                1.0,
            )

    return weights


def compute_ingredient_match(
    product_id: str,
    q: Dict[str, Any],
) -> Dict[str, Any]:
    profile = get_product_ingredient_profile(
        product_id
    )

    if not profile["available"]:
        return {
            "ingredient_data_available": False,
            "ingredient_match_score": 0.0,
            "ingredient_coverage": 0.0,
            "matched_ingredients": [],
            "ingredient_match_types": {},
            "ingredient_profile": profile,
        }

    desired = requested_ingredient_weights(q)

    if not desired:
        return {
            "ingredient_data_available": True,
            "ingredient_match_score": 0.0,
            "ingredient_coverage": 0.0,
            "matched_ingredients": [],
            "ingredient_match_types": {},
            "ingredient_profile": profile,
        }

    product_canonical = set(
        profile["canonical_ingredients"]
    )
    product_tokens = set(
        profile["raw_tokens"]
    )
    raw_text = profile["normalized_text"]

    matched = []
    match_types = {}
    weighted_match = 0.0
    total_weight = sum(
        desired.values()
    )

    for desired_name, relevance in desired.items():
        target = canonicalize_ingredient(
            desired_name
        )

        if not target:
            continue

        # Exact/canonical match.
        if target in product_canonical:
            matched.append(target)
            match_types[target] = "exact_or_canonical"
            weighted_match += float(relevance)
            continue

        # Alias match.
        aliases = INGREDIENT_ALIASES.get(
            target,
            [target],
        )

        alias_hit = False

        for alias in aliases:
            alias_norm = normalize_token(alias)

            if alias_norm in product_tokens:
                alias_hit = True
                break

            if alias_norm and re.search(
                rf"(?<![a-z0-9]){re.escape(alias_norm)}(?![a-z0-9])",
                raw_text,
            ):
                alias_hit = True
                break

        if alias_hit:
            matched.append(target)
            match_types[target] = "alias"
            weighted_match += float(relevance)

    matched = sorted(set(matched))

    coverage = (
        len(matched) / len(desired)
        if desired
        else 0.0
    )

    score = (
        weighted_match / total_weight
        if total_weight
        else 0.0
    )

    return {
        "ingredient_data_available": True,
        "ingredient_match_score": float(
            np.clip(score, 0.0, 1.0)
        ),
        "ingredient_coverage": float(
            np.clip(coverage, 0.0, 1.0)
        ),
        "matched_ingredients": matched,
        "ingredient_match_types": match_types,
        "ingredient_profile": profile,
    }


# Backward-compatible names used by the notebook/application.
def ingredient_match_profile(
    product_id: str,
    q: Dict[str, Any],
) -> Dict[str, Any]:
    return compute_ingredient_match(
        product_id,
        q,
    )


def ingredient_match_score(
    product_id: str,
    q: Dict[str, Any],
) -> Tuple[float, List[str]]:
    result = compute_ingredient_match(
        product_id,
        q,
    )
    return (
        result["ingredient_match_score"],
        result["matched_ingredients"],
    )


# Cache once. All later calls use the same canonical representation.
ingredient_profiles = {
    str(row["product_id"]): get_product_ingredient_profile(
        str(row["product_id"])
    )
    for _, row in catalog.iterrows()
}

print("Ingredient profiles built:", len(ingredient_profiles))
print(
    "Products with ingredient data:",
    sum(
        p["available"]
        for p in ingredient_profiles.values()
    ),
)
print(
    "Products without ingredient data:",
    sum(
        not p["available"]
        for p in ingredient_profiles.values()
    ),
)

assert len(ingredient_profiles) == len(catalog)
print("Ingredient profile creation: PASS")

Ingredient profiles built: 2420
Products with ingredient data: 2286
Products without ingredient data: 134
Ingredient profile creation: PASS


# 14. Ingredient Root-Cause Diagnostic

This diagnostic is specifically retained to prove that the original issue is fixed and that missing ingredient data is handled honestly.

In [22]:
diagnostic_query = normalize_query(
    "lightweight sunscreen for oily acne-prone skin with zinc oxide under $30"
)

diagnostic_ids = [
    "P454391",
    "P483658",
    "P500112",
]

diagnostic_rows = []

print("=" * 100)
print("INGREDIENT ROOT-CAUSE DIAGNOSTIC")
print("=" * 100)
print(
    "Query:",
    build_query_text(diagnostic_query),
)
print()

for pid in diagnostic_ids:
    result = compute_ingredient_match(
        pid,
        diagnostic_query,
    )

    row = catalog_by_id.loc[pid]

    diagnostic_rows.append({
        "product_id": pid,
        "product_name": safe_text(
            row.get("product_name")
        ),
        "ingredient_data_available": result[
            "ingredient_data_available"
        ],
        "ingredient_match_score": result[
            "ingredient_match_score"
        ],
        "ingredient_coverage": result[
            "ingredient_coverage"
        ],
        "matched_ingredients": result[
            "matched_ingredients"
        ],
        "match_types": result[
            "ingredient_match_types"
        ],
    })

    print(pid, "—", row.get("product_name"))
    print(
        "  available:",
        result["ingredient_data_available"],
    )
    print(
        "  canonical ingredients:",
        result["ingredient_profile"][
            "canonical_ingredients"
        ][:20],
    )
    print(
        "  matched:",
        result["matched_ingredients"],
    )
    print(
        "  score:",
        round(
            result["ingredient_match_score"],
            4,
        ),
    )
    print(
        "  coverage:",
        round(
            result["ingredient_coverage"],
            4,
        ),
    )
    print("-" * 100)

diagnostic_df = pd.DataFrame(
    diagnostic_rows
)

display(diagnostic_df)

# Explicit invariants.
p454 = compute_ingredient_match(
    "P454391",
    diagnostic_query,
)
p483 = compute_ingredient_match(
    "P483658",
    normalize_query("sunscreen containing zinc oxide"),
)

assert p454["ingredient_data_available"] is False
assert p454["ingredient_match_score"] == 0.0
assert p454["ingredient_coverage"] == 0.0
assert p454["matched_ingredients"] == []

assert p483["ingredient_data_available"] is True
assert p483["ingredient_match_score"] > 0.0
assert "zinc" in p483["matched_ingredients"]

print("Ingredient root-cause diagnostic: PASS")

INGREDIENT ROOT-CAUSE DIAGNOSTIC
Query: lightweight sunscreen for oily acne-prone skin with zinc oxide under $30 | skin type: oily | concerns: acne, oil_control | category: sunscreen | preferences: lightweight | ingredients: zinc

P454391 — PLAY Antioxidant Body Sunscreen Mist SPF 30 PA++++
  available: False
  canonical ingredients: []
  matched: []
  score: 0.0
  coverage: 0.0
----------------------------------------------------------------------------------------------------
P483658 — Mineral Sunscreen Zinc Oxide Broad Spectrum SPF 30
  available: True
  canonical ingredients: ['glycerin', 'zinc']
  matched: ['zinc']
  score: 0.2041
  coverage: 0.2
----------------------------------------------------------------------------------------------------
P500112 — N°41 Facial Sunscreen Mist with SPF 41
  available: True
  canonical ingredients: ['alcohol denat', 'glycerin']
  matched: []
  score: 0.0
  coverage: 0.0
--------------------------------------------------------------------------

,product_id,product_name,ingredient_data_available,ingredient_match_score,ingredient_coverage,matched_ingredients,match_types
0,P454391,PLAY Antioxidant Body Sunscreen Mist SPF 30 PA...,False,0.000000,0.0,[],{}
1,P483658,Mineral Sunscreen Zinc Oxide Broad Spectrum SP...,True,0.204082,0.2,[zinc],{'zinc': 'exact_or_canonical'}
2,P500112,N°41 Facial Sunscreen Mist with SPF 41,True,0.000000,0.0,[],{}


Ingredient root-cause diagnostic: PASS


# 15. Review Scoring

Review-derived signals are query-aware. Positive themes are rewarded, while negative themes can be inverted for preferences such as `non-greasy` and `fragrance-free`.

In [23]:
REVIEW_THEME_MAP = {
    "lightweight": [
        "theme_lightweight_share",
        "theme_lightweight_count",
    ],
    "greasy": [
        "theme_greasy_share",
        "theme_greasy_count",
    ],
    "hydrating": [
        "theme_hydrating_share",
        "theme_hydrating_count",
    ],
    "drying": [
        "theme_drying_share",
        "theme_drying_count",
    ],
    "fragrance": [
        "theme_fragrance_share",
        "theme_fragrance_count",
    ],
    "irritation": [
        "theme_irritation_share",
        "theme_irritation_count",
    ],
    "breakout": [
        "theme_breakout_share",
        "theme_breakout_count",
    ],
    "absorption": [
        "theme_absorption_share",
        "theme_absorption_count",
    ],
    "sticky": [
        "theme_sticky_share",
        "theme_sticky_count",
    ],
    "texture": [
        "theme_texture_share",
        "theme_texture_count",
    ],
    "effective": [
        "theme_effective_share",
        "theme_effective_count",
    ],
}


def get_theme_signal(
    row: pd.Series,
    theme: str,
) -> Optional[float]:
    for col in REVIEW_THEME_MAP.get(
        theme,
        [],
    ):
        if col not in row.index:
            continue

        value = pd.to_numeric(
            row[col],
            errors="coerce",
        )

        if pd.notna(value):
            if col.endswith("_count"):
                return float(
                    1.0 - math.exp(
                        -float(value) / 20.0
                    )
                )

            return float(
                np.clip(
                    value,
                    0.0,
                    1.0,
                )
            )

    return None


def review_relevance_score(
    product_id: str,
    q: Dict[str, Any],
) -> Tuple[float, List[str]]:
    pid = str(product_id)

    if pid not in catalog_by_id.index:
        return 0.0, []

    row = catalog_by_id.loc[pid]

    requests: List[
        Tuple[str, int]
    ] = []

    for pref in q.get(
        "effective_preferences",
        [],
    ):
        pref = normalize_token(pref)

        if pref == "lightweight":
            requests.append(
                ("lightweight", +1)
            )
        elif pref == "non_greasy":
            requests.append(
                ("greasy", -1)
            )
        elif pref == "fragrance_free":
            requests.append(
                ("fragrance", -1)
            )
        elif pref == "non_comedogenic":
            requests.append(
                ("breakout", +1)
            )

    concern_to_theme = {
        "hydration": ("hydrating", +1),
        "acne": ("breakout", +1),
        "oil_control": ("greasy", -1),
    }

    for concern in q.get(
        "effective_concerns",
        [],
    ):
        mapping = concern_to_theme.get(
            normalize_token(concern)
        )

        if mapping:
            requests.append(mapping)

    if not requests:
        generic = pd.to_numeric(
            row.get(
                "recommendation_rate",
                np.nan,
            ),
            errors="coerce",
        )

        if pd.notna(generic):
            return (
                float(
                    np.clip(
                        generic,
                        0.0,
                        1.0,
                    )
                ),
                ["overall recommendation rate"],
            )

        return 0.0, []

    scores = []
    evidence = []

    for theme, direction in requests:
        signal = get_theme_signal(
            row,
            theme,
        )

        if signal is None:
            continue

        value = (
            float(signal)
            if direction > 0
            else float(1.0 - signal)
        )

        scores.append(
            np.clip(value, 0.0, 1.0)
        )

        evidence.append(
            theme
        )

    if not scores:
        return 0.0, []

    return (
        float(np.mean(scores)),
        list(dict.fromkeys(evidence)),
    )


def add_review_scores(
    df: pd.DataFrame,
    q: Dict[str, Any],
) -> pd.DataFrame:
    out = df.copy()

    values = [
        review_relevance_score(
            str(pid),
            q,
        )
        for pid in out["product_id"]
    ]

    out["review_relevance_score"] = [
        v[0]
        for v in values
    ]
    out["review_evidence"] = [
        v[1]
        for v in values
    ]

    return out

# 16. Preference Scoring

Preference scoring combines skin-type audience signals with explicit product descriptors.

In [24]:
def skin_preference_score(
    row: pd.Series,
    skin_type: Optional[str],
) -> Tuple[float, bool]:
    if not skin_type:
        return 0.0, False

    col = (
        "skin_share_"
        + normalize_token(skin_type).replace(
            " ",
            "_",
        )
    )

    if col not in row.index:
        return 0.0, False

    value = pd.to_numeric(
        row[col],
        errors="coerce",
    )

    if pd.isna(value):
        return 0.0, False

    return (
        float(
            np.clip(
                value,
                0.0,
                1.0,
            )
        ),
        True,
    )


def descriptor_preference_score(
    row: pd.Series,
    preferences: Sequence[str],
) -> Tuple[float, bool]:
    if not preferences:
        return 0.0, False

    text = normalize_token(
        " ".join(
            safe_text(
                row.get(col, "")
            )
            for col in [
                "product_document",
                "highlights",
                "ingredients_clean",
                "ingredients",
            ]
        )
    )

    scores = []

    for pref in preferences:
        canonical_pref = normalize_token(
            pref
        )

        aliases = PREFERENCE_TERMS.get(
            canonical_pref,
            [canonical_pref],
        )

        scores.append(
            1.0
            if any(
                normalize_token(alias) in text
                for alias in aliases
            )
            else 0.0
        )

    return (
        float(np.mean(scores)),
        True,
    )


def preference_score(
    product_id: str,
    q: Dict[str, Any],
) -> Tuple[float, bool]:
    pid = str(product_id)

    if pid not in catalog_by_id.index:
        return 0.0, False

    row = catalog_by_id.loc[pid]

    signals = []

    skin_score, skin_available = (
        skin_preference_score(
            row,
            q.get("effective_skin_type"),
        )
    )

    if skin_available:
        signals.append(skin_score)

    descriptor_score, descriptor_available = (
        descriptor_preference_score(
            row,
            q.get("effective_preferences", []),
        )
    )

    if descriptor_available:
        signals.append(descriptor_score)

    if not signals:
        return 0.0, False

    return (
        float(np.mean(signals)),
        True,
    )


def add_preference_scores(
    df: pd.DataFrame,
    q: Dict[str, Any],
) -> pd.DataFrame:
    out = df.copy()

    values = [
        preference_score(
            str(pid),
            q,
        )
        for pid in out["product_id"]
    ]

    out["preference_match_score"] = [
        v[0]
        for v in values
    ]
    out["preference_signal_available"] = [
        v[1]
        for v in values
    ]

    return out

# 17. Rating-Quality Scoring

A raw star rating is combined with review volume using Bayesian-style shrinkage toward the catalog mean.

In [25]:
rating_series = pd.to_numeric(
    catalog.get(
        "review_avg_rating",
        catalog.get(
            "rating",
            pd.Series(dtype=float),
        ),
    ),
    errors="coerce",
).dropna()

RATING_GLOBAL_MEAN = (
    float(rating_series.mean())
    if not rating_series.empty
    else 3.5
)

RATING_PRIOR_COUNT = 50.0

print(
    "Rating prior mean:",
    round(RATING_GLOBAL_MEAN, 4),
)
print(
    "Rating prior count:",
    RATING_PRIOR_COUNT,
)


def rating_quality_score(
    product_id: str,
) -> float:
    pid = str(product_id)

    if pid not in catalog_by_id.index:
        return 0.0

    row = catalog_by_id.loc[pid]

    rating = numeric_value(
        row,
        [
            "review_avg_rating",
            "rating",
        ],
        default=np.nan,
    )

    count = numeric_value(
        row,
        [
            "review_count_observed",
            "reviews",
        ],
        default=0.0,
    )

    if not np.isfinite(rating):
        return 0.0

    adjusted = (
        count * rating
        + RATING_PRIOR_COUNT
        * RATING_GLOBAL_MEAN
    ) / (
        count
        + RATING_PRIOR_COUNT
    )

    return float(
        np.clip(
            (adjusted - 1.0) / 4.0,
            0.0,
            1.0,
        )
    )


def add_rating_scores(
    df: pd.DataFrame,
) -> pd.DataFrame:
    out = df.copy()

    out["rating_quality_score"] = [
        rating_quality_score(
            str(pid)
        )
        for pid in out["product_id"]
    ]

    return out

Rating prior mean: 4.2268
Rating prior count: 50.0


# 18. Diversity-aware Ranking

The final score uses exactly six signals.

```text
0.40 Semantic
0.25 Ingredient
0.15 Review
0.10 Preference
0.05 Rating
0.05 Diversity
```

These values are engineering heuristics, not claims of optimized weights.
For products without ingredient data, the available weights are renormalized so missing source data is not treated as negative evidence.

In [26]:
RANKING_WEIGHTS = {
    "semantic": 0.40,
    "ingredient": 0.25,
    "review": 0.15,
    "preference": 0.10,
    "rating": 0.05,
    "diversity": 0.05,
}

print("Ranking weights:")
display(pd.DataFrame([
    {
        "signal": key,
        "weight": value,
    }
    for key, value in RANKING_WEIGHTS.items()
]))

assert math.isclose(
    sum(RANKING_WEIGHTS.values()),
    1.0,
    abs_tol=1e-9,
)
print("Weight validation: PASS")

Ranking weights:


,signal,weight
0,semantic,0.40
1,ingredient,0.25
2,review,0.15
3,preference,0.10
4,rating,0.05
5,diversity,0.05


Weight validation: PASS


In [27]:
embedding_position = {
    str(pid): idx
    for idx, pid in enumerate(
        catalog["product_id"].astype(str)
    )
}


def embedding_similarity(
    product_a: str,
    product_b: str,
) -> float:
    ia = embedding_position.get(
        str(product_a)
    )
    ib = embedding_position.get(
        str(product_b)
    )

    if ia is None or ib is None:
        return 0.0

    return float(
        np.clip(
            np.dot(
                product_embeddings[ia],
                product_embeddings[ib],
            ),
            -1.0,
            1.0,
        )
    )


def diversity_score_for_candidate(
    row: pd.Series,
    selected_rows: Sequence[pd.Series],
) -> float:
    if not selected_rows:
        return 1.0

    pid = str(row["product_id"])

    max_embedding_similarity = max(
        embedding_similarity(
            pid,
            str(other["product_id"]),
        )
        for other in selected_rows
    )

    candidate_brand = normalize_token(
        row.get("brand_name", "")
    )
    candidate_category = normalize_token(
        row.get(
            "secondary_category",
            row.get(
                "primary_category",
                "",
            ),
        )
    )

    repetition_penalty = 0.0

    for other in selected_rows:
        if (
            candidate_brand
            and candidate_brand
            == normalize_token(
                other.get("brand_name", "")
            )
        ):
            repetition_penalty = max(
                repetition_penalty,
                0.10,
            )

        if (
            candidate_category
            and candidate_category
            == normalize_token(
                other.get(
                    "secondary_category",
                    other.get(
                        "primary_category",
                        "",
                    ),
                )
            )
        ):
            repetition_penalty = max(
                repetition_penalty,
                0.05,
            )

    return float(
        np.clip(
            1.0
            - (
                0.85
                * max_embedding_similarity
                + repetition_penalty
            ),
            0.0,
            1.0,
        )
    )


def final_score(
    row: pd.Series,
    diversity: float,
) -> float:
    components = {
        "semantic": float(
            np.clip(
                row.get(
                    "semantic_similarity",
                    0.0,
                ),
                0.0,
                1.0,
            )
        ),
        "ingredient": float(
            np.clip(
                row.get(
                    "ingredient_match_score",
                    0.0,
                ),
                0.0,
                1.0,
            )
        ),
        "review": float(
            np.clip(
                row.get(
                    "review_relevance_score",
                    0.0,
                ),
                0.0,
                1.0,
            )
        ),
        "preference": float(
            np.clip(
                row.get(
                    "preference_match_score",
                    0.0,
                ),
                0.0,
                1.0,
            )
        ),
        "rating": float(
            np.clip(
                row.get(
                    "rating_quality_score",
                    0.0,
                ),
                0.0,
                1.0,
            )
        ),
        "diversity": float(
            np.clip(
                diversity,
                0.0,
                1.0,
            )
        ),
    }

    weights = dict(RANKING_WEIGHTS)

    # Missing ingredient data is excluded from the denominator
    # instead of becoming a false negative.
    if not bool(
        row.get(
            "ingredient_data_available",
            True,
        )
    ):
        weights.pop(
            "ingredient",
            None,
        )

    denominator = sum(
        weights.values()
    )

    normalized_weights = {
        key: weight / denominator
        for key, weight in weights.items()
    }

    score = sum(
        normalized_weights[key]
        * components[key]
        for key in normalized_weights
    )

    return float(
        np.clip(
            score,
            0.0,
            1.0,
        )
    )


def rank_candidates(
    df: pd.DataFrame,
    top_k: int = 5,
) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    working = df.copy().reset_index(
        drop=True
    )

    selected_indices = []
    selected_rows: List[pd.Series] = []
    final_scores_map = {}
    diversity_map = {}

    target = min(
        int(top_k),
        len(working),
    )

    for _ in range(target):
        best_idx = None
        best_score = -float("inf")
        best_diversity = 0.0

        for idx, row in working.iterrows():
            if idx in selected_indices:
                continue

            diversity = diversity_score_for_candidate(
                row,
                selected_rows,
            )

            score = final_score(
                row,
                diversity,
            )

            if score > best_score:
                best_idx = idx
                best_score = score
                best_diversity = diversity

        if best_idx is None:
            break

        selected_indices.append(best_idx)
        selected_rows.append(
            working.loc[best_idx]
        )
        final_scores_map[best_idx] = best_score
        diversity_map[best_idx] = best_diversity

    ranked = working.loc[
        selected_indices
    ].copy()

    ranked["diversity_score"] = [
        diversity_map[idx]
        for idx in selected_indices
    ]

    ranked["final_score"] = [
        final_scores_map[idx]
        for idx in selected_indices
    ]

    return ranked.sort_values(
        "final_score",
        ascending=False,
    ).reset_index(drop=True)

# 19. Deterministic Explainability

Every explanation is generated from fields actually used by the ranking pipeline.

In [28]:
def explanation_for_product(
    row: pd.Series,
    q: Dict[str, Any],
) -> Dict[str, Any]:
    reasons = []

    requested_category = normalize_token(
        q.get("effective_category")
    )

    category_text = normalize_token(
        " ".join([
            safe_text(
                row.get(
                    "primary_category",
                    "",
                )
            ),
            safe_text(
                row.get(
                    "secondary_category",
                    "",
                )
            ),
            safe_text(
                row.get(
                    "tertiary_category",
                    "",
                )
            ),
        ])
    )

    if (
        requested_category
        and requested_category in category_text
    ):
        reasons.append(
            f"Matches requested {requested_category} category"
        )

    budget = q.get(
        "effective_budget_max"
    )
    price = pd.to_numeric(
        row.get(
            "effective_price_usd",
            np.nan,
        ),
        errors="coerce",
    )

    if (
        budget is not None
        and pd.notna(price)
        and price <= float(budget)
    ):
        reasons.append(
            f"Within budget (${float(price):.2f} ≤ ${float(budget):.2f})"
        )

    semantic = float(
        row.get(
            "semantic_similarity",
            0.0,
        )
    )

    if semantic >= 0.75:
        reasons.append(
            "Strong semantic match"
        )
    elif semantic >= 0.55:
        reasons.append(
            "Good semantic match"
        )

    available = bool(
        row.get(
            "ingredient_data_available",
            False,
        )
    )

    matched = row.get(
        "matched_ingredients",
        row.get(
            "matched_ingredient_terms",
            [],
        ),
    )

    if isinstance(
        matched,
        str,
    ):
        try:
            matched = ast.literal_eval(
                matched
            )
        except Exception:
            matched = [
                matched
            ]

    matched = list(
        matched
    )

    coverage = float(
        row.get(
            "ingredient_coverage",
            0.0,
        )
    )

    if available and matched:
        reasons.append(
            "Ingredient signals matched: "
            + ", ".join(
                matched[:5]
            )
        )
        reasons.append(
            f"Ingredient coverage: {coverage:.0%}"
        )
    elif not available:
        reasons.append(
            "Ingredient information unavailable in catalog"
        )

    evidence = row.get(
        "review_evidence",
        [],
    )

    if isinstance(
        evidence,
        str,
    ):
        try:
            evidence = ast.literal_eval(
                evidence
            )
        except Exception:
            evidence = [evidence]

    if evidence:
        reasons.append(
            "Review evidence: "
            + ", ".join(
                str(x).replace(
                    "_",
                    " ",
                )
                for x in list(evidence)[:4]
            )
        )

    skin = normalize_token(
        q.get("effective_skin_type")
    )

    if skin:
        skin_col = (
            f"skin_share_{skin.replace(' ', '_')}"
        )

        if skin_col in row.index:
            skin_share = pd.to_numeric(
                row.get(skin_col),
                errors="coerce",
            )

            if (
                pd.notna(skin_share)
                and float(skin_share) > 0
            ):
                reasons.append(
                    f"Review audience includes {skin} skin "
                    f"({float(skin_share):.0%})"
                )

    if float(
        row.get(
            "rating_quality_score",
            0.0,
        )
    ) >= 0.80:
        reasons.append(
            "Strong rating quality"
        )

    if not reasons:
        reasons.append(
            "Selected from the highest-scoring valid candidates"
        )

    return {
        "reasons": reasons[:7],
        "ingredient_data_available": available,
        "matched_ingredients": matched[:10],
        "ingredient_coverage": round(
            coverage,
            4,
        ),
        "ingredient_match_types": row.get(
            "ingredient_match_types",
            {},
        ),
        "review_evidence": list(
            evidence
        )[:10],
        "score_breakdown": {
            "semantic_similarity": round(
                float(
                    row.get(
                        "semantic_similarity",
                        0.0,
                    )
                ),
                4,
            ),
            "ingredient_match": round(
                float(
                    row.get(
                        "ingredient_match_score",
                        0.0,
                    )
                ),
                4,
            ),
            "ingredient_coverage": round(
                coverage,
                4,
            ),
            "review_relevance": round(
                float(
                    row.get(
                        "review_relevance_score",
                        0.0,
                    )
                ),
                4,
            ),
            "preference_match": round(
                float(
                    row.get(
                        "preference_match_score",
                        0.0,
                    )
                ),
                4,
            ),
            "rating_quality": round(
                float(
                    row.get(
                        "rating_quality_score",
                        0.0,
                    )
                ),
                4,
            ),
            "diversity": round(
                float(
                    row.get(
                        "diversity_score",
                        0.0,
                    )
                ),
                4,
            ),
            "final_score": round(
                float(
                    row.get(
                        "final_score",
                        0.0,
                    )
                ),
                4,
            ),
        },
    }

# 20. Build the Final `recommend()` Function

This is the single application-facing function.

The future Flask service should import/call this function rather than duplicate ranking logic.

In [30]:
# ============================================================
# 20. Build the Final recommend() Function
# ============================================================

def add_ingredient_scores(
    df: pd.DataFrame,
    q: Dict[str, Any],
) -> pd.DataFrame:
    """
    Add the canonical ingredient-matching results to each candidate.

    This function uses the same compute_ingredient_match() function
    used by the ingredient diagnostics, so the recommendation results
    cannot drift from the diagnostic results.
    """
    out = df.copy()

    results = []

    for product_id in out["product_id"].astype(str):
        result = compute_ingredient_match(
            product_id,
            q,
        )
        results.append(result)

    out["ingredient_data_available"] = [
        result["ingredient_data_available"]
        for result in results
    ]

    out["ingredient_match_score"] = [
        float(result["ingredient_match_score"])
        for result in results
    ]

    out["ingredient_coverage"] = [
        float(result["ingredient_coverage"])
        for result in results
    ]

    out["matched_ingredients"] = [
        result["matched_ingredients"]
        for result in results
    ]

    # Keep the old column name too for compatibility with any
    # earlier diagnostic/display cells.
    out["matched_ingredient_terms"] = [
        result["matched_ingredients"]
        for result in results
    ]

    out["ingredient_match_types"] = [
        result["ingredient_match_types"]
        for result in results
    ]

    return out


print("Ingredient scoring helper loaded successfully.")
print("Testing helper on current candidate set...")

# Run only when filtered_example exists from the previous test section.
if "filtered_example" in globals() and not filtered_example.empty:
    filtered_example = add_ingredient_scores(
        filtered_example,
        normalized_example,
    )

    print("Ingredient scoring result:")
    display(
        filtered_example[
            [
                "product_id",
                "product_name",
                "ingredient_data_available",
                "ingredient_match_score",
                "ingredient_coverage",
                "matched_ingredients",
                "ingredient_match_types",
            ]
        ].head(10)
    )

print("\n" + "=" * 90)
print("FINAL recommend() FUNCTION")
print("=" * 90)


def recommend(
    query: Optional[str] = None,
    skin_type: Optional[str] = None,
    concerns: Optional[Sequence[str]] = None,
    category: Optional[str] = None,
    budget_max: Optional[float] = None,
    preferred_terms: Optional[Sequence[str]] = None,
    avoid_ingredients: Optional[Sequence[str]] = None,
    candidate_k: int = 50,
    top_k: int = 5,
) -> Dict[str, Any]:

    started = time.perf_counter()

    # --------------------------------------------------------
    # 1. Normalize user input
    # --------------------------------------------------------
    q = normalize_query(
        query=query,
        skin_type=skin_type,
        concerns=concerns,
        category=category,
        budget_max=budget_max,
        preferred_terms=preferred_terms,
        avoid_ingredients=avoid_ingredients,
    )

    query_text = build_query_text(q)

    if not query_text:
        raise ValueError(
            "Please provide a query or at least one "
            "structured recommendation signal."
        )

    # --------------------------------------------------------
    # 2. Semantic candidate retrieval
    # --------------------------------------------------------
    retrieval_started = time.perf_counter()

    retrieved = semantic_retrieve(
        q,
        candidate_k=candidate_k,
    )

    retrieval_ms = (
        time.perf_counter()
        - retrieval_started
    ) * 1000

    # --------------------------------------------------------
    # 3. Hard constraints
    # --------------------------------------------------------
    filtered, filter_stats = apply_hard_filters(
        retrieved,
        q,
    )

    # --------------------------------------------------------
    # 4. No valid candidates
    # --------------------------------------------------------
    if filtered.empty:

        total_ms = (
            time.perf_counter()
            - started
        ) * 1000

        return {
            "status": "no_high_confidence_match",
            "query": q,
            "query_text": query_text,
            "recommendations": [],
            "filter_stats": filter_stats,
            "latency_ms": {
                "retrieval": round(
                    retrieval_ms,
                    2,
                ),
                "total": round(
                    total_ms,
                    2,
                ),
            },
            "message": (
                "No products satisfied the requested constraints. "
                "Try relaxing the budget, category, or ingredient constraints."
            ),
        }

    # --------------------------------------------------------
    # 5. Ingredient scoring
    # --------------------------------------------------------
    filtered = add_ingredient_scores(
        filtered,
        q,
    )

    # --------------------------------------------------------
    # 6. Review scoring
    # --------------------------------------------------------
    filtered = add_review_scores(
        filtered,
        q,
    )

    # --------------------------------------------------------
    # 7. Preference scoring
    # --------------------------------------------------------
    filtered = add_preference_scores(
        filtered,
        q,
    )

    # --------------------------------------------------------
    # 8. Rating quality
    # --------------------------------------------------------
    filtered = add_rating_scores(
        filtered,
    )

    # --------------------------------------------------------
    # 9. Final ranking
    # --------------------------------------------------------
    ranked = rank_candidates(
        filtered,
        top_k=top_k,
    )

    # --------------------------------------------------------
    # 10. Build clean recommendation response
    # --------------------------------------------------------
    recommendations = []

    for _, row in ranked.iterrows():

        explanation = explanation_for_product(
            row,
            q,
        )

        price = pd.to_numeric(
            row.get(
                "effective_price_usd",
                np.nan,
            ),
            errors="coerce",
        )

        recommendations.append(
            {
                "product_id": str(
                    row["product_id"]
                ),

                "product_name": safe_text(
                    row.get(
                        "product_name"
                    )
                ),

                "brand_name": safe_text(
                    row.get(
                        "brand_name"
                    )
                ),

                "primary_category": safe_text(
                    row.get(
                        "primary_category"
                    )
                ),

                "secondary_category": safe_text(
                    row.get(
                        "secondary_category"
                    )
                ),

                "price_usd": (
                    float(price)
                    if pd.notna(price)
                    else None
                ),

                "final_score": round(
                    float(
                        row["final_score"]
                    ),
                    4,
                ),

                "ingredient_data_available": (
                    explanation[
                        "ingredient_data_available"
                    ]
                ),

                "matched_ingredients": (
                    explanation[
                        "matched_ingredients"
                    ]
                ),

                "ingredient_coverage": (
                    explanation[
                        "ingredient_coverage"
                    ]
                ),

                "ingredient_match_types": (
                    explanation[
                        "ingredient_match_types"
                    ]
                ),

                "review_evidence": (
                    explanation[
                        "review_evidence"
                    ]
                ),

                "reasons": (
                    explanation[
                        "reasons"
                    ]
                ),

                "score_breakdown": (
                    explanation[
                        "score_breakdown"
                    ]
                ),
            }
        )

    total_ms = (
        time.perf_counter()
        - started
    ) * 1000

    return {
        "status": "ok",
        "query": q,
        "query_text": query_text,
        "recommendations": recommendations,
        "filter_stats": filter_stats,
        "latency_ms": {
            "retrieval": round(
                retrieval_ms,
                2,
            ),
            "total": round(
                total_ms,
                2,
            ),
        },
    }


# ============================================================
# Demo execution
# ============================================================

demo_result = recommend(
    query="lightweight moisturizer for dry skin",
    top_k=5,
)

print("\nFINAL DEMO RESULT")
print("=" * 90)

print(
    "Status:",
    demo_result["status"],
)

print(
    "Query:",
    demo_result["query_text"],
)

print(
    "Recommendations:",
    len(
        demo_result[
            "recommendations"
        ]
    ),
)

print(
    "Total latency (ms):",
    demo_result[
        "latency_ms"
    ]["total"],
)

for rank, item in enumerate(
    demo_result[
        "recommendations"
    ],
    start=1,
):

    print(
        f"\n{rank}. "
        f"{item['product_name']} "
        f"(score={item['final_score']:.4f})"
    )

    print(
        "   Brand:",
        item["brand_name"],
    )

    print(
        "   Price:",
        item["price_usd"],
    )

    print(
        "   Ingredient data:",
        item[
            "ingredient_data_available"
        ],
    )

    print(
        "   Matched ingredients:",
        item[
            "matched_ingredients"
        ],
    )

    print(
        "   Ingredient coverage:",
        item[
            "ingredient_coverage"
        ],
    )

    print(
        "   Reasons:",
        " | ".join(
            item[
                "reasons"
            ][:4]
        ),
    )

print("\nrecommend() function: READY")

Ingredient scoring helper loaded successfully.
Testing helper on current candidate set...

FINAL recommend() FUNCTION

FINAL DEMO RESULT
Status: ok
Query: lightweight moisturizer for dry skin | skin type: dry | category: moisturizer | preferences: lightweight
Recommendations: 5
Total latency (ms): 94.73

1. GinZing Ultra-Hydrating Energy-Boosting Cream (score=0.5793)
   Brand: Origins
   Price: 36.0
   Ingredient data: False
   Matched ingredients: []
   Ingredient coverage: 0.0
   Reasons: Matches requested moisturizer category | Strong semantic match | Ingredient information unavailable in catalog | Review evidence: lightweight

2. Regenerative Anti-Aging Moisturizer (score=0.5147)
   Brand: Algenist
   Price: 94.0
   Ingredient data: False
   Matched ingredients: []
   Ingredient coverage: 0.0
   Reasons: Matches requested moisturizer category | Strong semantic match | Ingredient information unavailable in catalog | Review evidence: lightweight

3. Nutritious Super-Pomegranate Radia

# 21. Recommendation Test Suite

In [31]:
TEST_CASES = [
    {
        "name": "Oily acne-prone sunscreen",
        "kwargs": {
            "query": (
                "lightweight sunscreen for oily "
                "acne-prone skin under $30"
            ),
            "top_k": 5,
        },
    },
    {
        "name": "Dry skin moisturizer",
        "kwargs": {
            "skin_type": "dry",
            "concerns": ["hydration"],
            "category": "moisturizer",
            "budget_max": 40,
            "preferred_terms": ["hydrating"],
            "top_k": 5,
        },
    },
    {
        "name": "Acne treatment",
        "kwargs": {
            "query": (
                "acne treatment with ingredients "
                "associated with breakout care"
            ),
            "top_k": 5,
        },
    },
    {
        "name": "Budget cleanser",
        "kwargs": {
            "category": "cleanser",
            "budget_max": 20,
            "top_k": 5,
        },
    },
    {
        "name": "Sensitive fragrance-free moisturizer",
        "kwargs": {
            "query": (
                "fragrance-free moisturizer "
                "for sensitive skin"
            ),
            "top_k": 5,
        },
    },
]

test_summary = []

for case in TEST_CASES:
    print("=" * 100)
    print(case["name"])
    print("=" * 100)

    result = recommend(
        **case["kwargs"]
    )

    print(
        "Status:",
        result["status"],
    )
    print(
        "Candidates after hard filters:",
        result["filter_stats"].get(
            "after_filter"
        ),
    )

    rows = []

    for rank, item in enumerate(
        result["recommendations"],
        start=1,
    ):
        rows.append({
            "rank": rank,
            "product_id": item["product_id"],
            "product": item["product_name"],
            "brand": item["brand_name"],
            "price": item["price_usd"],
            "score": item["final_score"],
            "ingredients": ", ".join(
                item["matched_ingredients"]
            ),
            "reasons": " | ".join(
                item["reasons"][:3]
            ),
        })

    if rows:
        display(
            pd.DataFrame(rows)
        )

    test_summary.append({
        "test": case["name"],
        "status": result["status"],
        "recommendations": len(
            result["recommendations"]
        ),
        "latency_ms": result["latency_ms"][
            "total"
        ],
    })

print("\nTest summary:")
display(
    pd.DataFrame(test_summary)
)

Oily acne-prone sunscreen
Status: ok
Candidates after hard filters: 6


,rank,product_id,product,brand,price,score,ingredients,reasons
0,1,P454391,PLAY Antioxidant Body Sunscreen Mist SPF 30 PA...,Supergoop!,21.0,0.5965,,Matches requested sunscreen category | Within ...
1,2,P483658,Mineral Sunscreen Zinc Oxide Broad Spectrum SP...,First Aid Beauty,28.0,0.4416,zinc,Matches requested sunscreen category | Within ...
2,3,P482322,Mini Mineral Mattescreen Sunscreen SPF 40 PA+++,Supergoop!,22.0,0.4317,zinc,Matches requested sunscreen category | Within ...
3,4,P482321,Mini Mineral Sheerscreen Sunscreen SPF 30 PA+++,Supergoop!,22.0,0.4281,zinc,Matches requested sunscreen category | Within ...
4,5,P500112,N°41 Facial Sunscreen Mist with SPF 41,HABIT,30.0,0.3972,,Matches requested sunscreen category | Within ...


Dry skin moisturizer
Status: ok
Candidates after hard filters: 9


,rank,product_id,product,brand,price,score,ingredients,reasons
0,1,P427641,Mini Confidence in a Cream Hydrating Moisturizer,IT Cosmetics,20.0,0.8093,"ceramide, glycerin, hyaluronic acid, squalane",Matches requested moisturizer category | Withi...
1,2,P500094,Hydrating Oil-Free Gel Moisturizer,Community Sixty-Six,25.0,0.7360,"glycerin, hyaluronic acid, squalane",Matches requested moisturizer category | Withi...
2,3,P378852,GinZing Ultra-Hydrating Energy-Boosting Cream,Origins,36.0,0.7277,,Matches requested moisturizer category | Withi...
3,4,P480280,Mini Squalane + Omega Repair Deep Hydration Mo...,Biossance,20.0,0.7137,"ceramide, glycerin, hyaluronic acid, squalane",Matches requested moisturizer category | Withi...
4,5,P2043,Brumisateur Natural Mineral Water Facial Spray,Evian,14.0,0.6154,,Matches requested moisturizer category | Withi...


Acne treatment
Status: ok
Candidates after hard filters: 32


...

,rank,product_id,product,brand,price,score,ingredients,reasons
0,1,P454771,Aglow Cleansing Butter,lilah b.,7.00,0.6783,,Matches requested cleanser category | Within b...
1,2,P473726,Smoothing Cleanser,SEPHORA COLLECTION,12.00,0.4862,,Matches requested cleanser category | Within b...
2,3,P438618,Clean Skin Gel Cleanser with Prebiotics,SEPHORA COLLECTION,12.00,0.4802,,Matches requested cleanser category | Within b...
3,4,P500113,Detoxifying AHA/BHA Gel Cleanser,Community Sixty-Six,16.00,0.4682,,Matches requested cleanser category | Within b...
4,5,P479353,Hyaluronic Acid Cleanser,The INKEY List,10.99,0.4577,,Matches requested cleanser category | Within b...


Sensitive fragrance-free moisturizer
Status: ok
Candidates after hard filters: 31


,rank,product_id,product,brand,price,score,ingredients,reasons
0,1,P483693,Essential Comfort Moisture Cream,Sulwhasoo,80.0,0.7941,fragrance,Matches requested moisturizer category | Stron...
1,2,P201440,Redness Solutions with Probiotic Technology Da...,CLINIQUE,56.0,0.7311,,Matches requested moisturizer category | Stron...
2,3,P505160,The Moisturizing Soft Cream Moisturizer,La Mer,380.0,0.7166,fragrance,Matches requested moisturizer category | Stron...
3,4,P411403,Confidence in a Cream Anti-Aging Hydrating Moi...,IT Cosmetics,20.0,0.7143,fragrance,Matches requested moisturizer category | Stron...
4,5,P427641,Mini Confidence in a Cream Hydrating Moisturizer,IT Cosmetics,20.0,0.6954,fragrance,Matches requested moisturizer category | Stron...



Test summary:


,test,status,recommendations,latency_ms
0,Oily acne-prone sunscreen,ok,5,47.90
1,Dry skin moisturizer,ok,5,47.16
2,Acne treatment,ok,5,124.76
3,Budget cleanser,ok,5,44.05
4,Sensitive fragrance-free moisturizer,ok,5,138.45


# 22. Ingredient Sanity Tests

These tests are intentionally deterministic so we can prove the ingredient subsystem is behaving correctly before final validation.

In [32]:
ingredient_tests = [
    (
        "zinc oxide",
        "P483658",
    ),
    (
        "zinc",
        "P483658",
    ),
    (
        "niacinamide",
        "P483658",
    ),
    (
        "salicylic acid",
        "P500112",
    ),
]

print("=" * 100)
print("INGREDIENT SANITY TESTS")
print("=" * 100)

for ingredient, pid in ingredient_tests:
    q = normalize_query(
        f"skincare product containing {ingredient}"
    )

    result = compute_ingredient_match(
        pid,
        q,
    )

    print(
        f"{ingredient:20s} + {pid}: "
        f"available={result['ingredient_data_available']}, "
        f"score={result['ingredient_match_score']:.4f}, "
        f"coverage={result['ingredient_coverage']:.4f}, "
        f"matched={result['matched_ingredients']}, "
        f"type={result['ingredient_match_types']}"
    )

zinc = compute_ingredient_match(
    "P483658",
    normalize_query(
        "sunscreen containing zinc oxide"
    ),
)

missing = compute_ingredient_match(
    "P454391",
    normalize_query(
        "sunscreen containing zinc"
    ),
)

assert zinc["ingredient_data_available"] is True
assert zinc["ingredient_match_score"] > 0
assert "zinc" in zinc[
    "matched_ingredients"
]

assert missing[
    "ingredient_data_available"
] is False

assert missing[
    "ingredient_match_score"
] == 0

assert missing[
    "ingredient_coverage"
] == 0

assert missing[
    "matched_ingredients"
] == []

print(
    "\nIngredient sanity tests: PASS"
)

INGREDIENT SANITY TESTS
zinc oxide           + P483658: available=True, score=1.0000, coverage=1.0000, matched=['zinc'], type={'zinc': 'exact_or_canonical'}
zinc                 + P483658: available=True, score=1.0000, coverage=1.0000, matched=['zinc'], type={'zinc': 'exact_or_canonical'}
niacinamide          + P483658: available=True, score=0.0000, coverage=0.0000, matched=[], type={}
salicylic acid       + P500112: available=True, score=0.0000, coverage=0.0000, matched=[], type={}

Ingredient sanity tests: PASS


# 23. Edge Cases

In [33]:
EDGE_CASES = [
    (
        "Empty input",
        {},
    ),
    (
        "Impossible budget",
        {
            "query": "sunscreen for oily skin",
            "budget_max": 0.01,
            "top_k": 5,
        },
    ),
    (
        "Unknown avoided ingredient",
        {
            "query": "moisturizer for dry skin",
            "avoid_ingredients": [
                "ingredient-that-does-not-exist"
            ],
            "top_k": 5,
        },
    ),
    (
        "Extremely restrictive request",
        {
            "query": (
                "very specific sunscreen "
                "combination with no likely match"
            ),
            "category": "sunscreen",
            "budget_max": 1,
            "avoid_ingredients": ["water"],
            "top_k": 5,
        },
    ),
    (
        "Category-only query",
        {
            "category": "cleanser",
            "top_k": 5,
        },
    ),
]

edge_results = []

for name, kwargs in EDGE_CASES:
    print("=" * 100)
    print(name)
    print("=" * 100)

    try:
        result = recommend(
            **kwargs
        )

        print(
            "Status:",
            result["status"],
        )
        print(
            "Recommendations:",
            len(result["recommendations"]),
        )

        edge_results.append({
            "case": name,
            "status": result["status"],
            "recommendations": len(
                result["recommendations"]
            ),
            "error": "",
        })

    except ValueError as exc:
        # Invalid user input should be a controlled validation error.
        print(
            "Controlled input validation:",
            str(exc),
        )

        edge_results.append({
            "case": name,
            "status": "controlled_validation",
            "recommendations": 0,
            "error": str(exc),
        })

    except Exception as exc:
        raise RuntimeError(
            f"Unexpected engine error in edge case '{name}': {exc}"
        ) from exc

display(
    pd.DataFrame(edge_results)
)

assert (
    edge_results[0]["status"]
    == "controlled_validation"
)

print(
    "Edge-case handling: PASS"
)

Empty input
Controlled input validation: Please provide a query or at least one structured recommendation signal.
Impossible budget
Status: no_high_confidence_match
Recommendations: 0
Unknown avoided ingredient
Status: ok
Recommendations: 5
Extremely restrictive request
Status: no_high_confidence_match
Recommendations: 0
Category-only query
Status: ok
Recommendations: 5


,case,status,recommendations,error
0,Empty input,controlled_validation,0,Please provide a query or at least one structu...
1,Impossible budget,no_high_confidence_match,0,
2,Unknown avoided ingredient,ok,5,
3,Extremely restrictive request,no_high_confidence_match,0,
4,Category-only query,ok,5,


Edge-case handling: PASS


# 24. Ranking Weight Sensitivity Check

Because the weights are engineering heuristics, we perform a small sensitivity check rather than claiming they are optimized.

The test varies the ingredient weight while preserving the relative proportions of the remaining signals.

In [34]:
def sensitivity_rank_scores(
    df: pd.DataFrame,
    ingredient_weight: float,
) -> pd.DataFrame:
    ingredient_weight = float(
        np.clip(
            ingredient_weight,
            0.0,
            1.0,
        )
    )

    remaining = 1.0 - ingredient_weight

    base_other = {
        "semantic": 0.40,
        "review": 0.15,
        "preference": 0.10,
        "rating": 0.05,
        "diversity": 0.05,
    }

    other_sum = sum(
        base_other.values()
    )

    scale = (
        remaining / other_sum
        if other_sum
        else 0.0
    )

    weights = {
        "semantic": base_other["semantic"] * scale,
        "ingredient": ingredient_weight,
        "review": base_other["review"] * scale,
        "preference": base_other["preference"] * scale,
        "rating": base_other["rating"] * scale,
        "diversity": base_other["diversity"] * scale,
    }

    rows = []

    for _, row in df.iterrows():
        components = {
            key: float(
                np.clip(
                    row.get(
                        {
                            "semantic": "semantic_similarity",
                            "ingredient": "ingredient_match_score",
                            "review": "review_relevance_score",
                            "preference": "preference_match_score",
                            "rating": "rating_quality_score",
                        }.get(key, ""),
                        0.0,
                    ),
                    0.0,
                    1.0,
                )
            )
            for key in [
                "semantic",
                "ingredient",
                "review",
                "preference",
                "rating",
            ]
        }

        components["diversity"] = 1.0

        if not bool(
            row.get(
                "ingredient_data_available",
                True,
            )
        ):
            weights_for_product = dict(weights)
            weights_for_product.pop(
                "ingredient",
                None,
            )
        else:
            weights_for_product = dict(weights)

        denom = sum(
            weights_for_product.values()
        )

        score = (
            sum(
                weights_for_product[k]
                * components[k]
                for k in weights_for_product
            )
            / denom
            if denom
            else 0.0
        )

        rows.append({
            "product_id": row["product_id"],
            "product_name": row.get(
                "product_name",
                "",
            ),
            "score": score,
        })

    return pd.DataFrame(rows)


sensitivity_q = normalize_query(
    "lightweight sunscreen for oily acne-prone skin under $30"
)

sens_retrieved = semantic_retrieve(
    sensitivity_q,
    candidate_k=30,
)

sens_filtered, _ = apply_hard_filters(
    sens_retrieved,
    sensitivity_q,
)

sens_filtered = add_ingredient_scores(
    sens_filtered,
    sensitivity_q,
)

sens_filtered = add_review_scores(
    sens_filtered,
    sensitivity_q,
)

sens_filtered = add_preference_scores(
    sens_filtered,
    sensitivity_q,
)

sens_filtered = add_rating_scores(
    sens_filtered,
)

sens_filtered["diversity_score"] = 1.0

sensitivity_outputs = []

if not sens_filtered.empty:
    for weight in [
        0.15,
        0.25,
        0.40,
    ]:
        temp = sensitivity_rank_scores(
            sens_filtered,
            weight,
        ).sort_values(
            "score",
            ascending=False,
        ).head(5)

        temp = temp.assign(
            ingredient_weight=weight
        )
        sensitivity_outputs.append(
            temp
        )

    sensitivity_df = pd.concat(
        sensitivity_outputs,
        ignore_index=True,
    )

    display(sensitivity_df)
else:
    print(
        "No candidates survived hard filters; sensitivity check is not applicable."
    )

print("Sensitivity analysis: PASS")

,product_id,product_name,score,ingredient_weight
0,P454391,PLAY Antioxidant Body Sunscreen Mist SPF 30 PA...,0.596531,0.15
1,P483658,Mineral Sunscreen Zinc Oxide Broad Spectrum SP...,0.524214,0.15
2,P454391,PLAY Antioxidant Body Sunscreen Mist SPF 30 PA...,0.596531,0.25
3,P483658,Mineral Sunscreen Zinc Oxide Broad Spectrum SP...,0.480444,0.25
4,P454391,PLAY Antioxidant Body Sunscreen Mist SPF 30 PA...,0.596531,0.40
5,P483658,Mineral Sunscreen Zinc Oxide Broad Spectrum SP...,0.414790,0.40


Sensitivity analysis: PASS


# 25. Save Application Artifacts

These artifacts are the hand-off from notebook development to the Flask/Streamlit application.

ENGINE_CONFIG_PATH = ARTIFACT_DIR / "recommendation_engine_config.json"
INGREDIENT_PROFILES_PATH = ARTIFACT_DIR / "ingredient_profiles.jsonl"
APPLICATION_CATALOG_PATH = ARTIFACT_DIR / "application_catalog.csv"

ENGINE_CONFIG = {
    "project": "DermaMatch AI",
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": EMBEDDING_DIM,
    "chroma_collection": CHROMA_COLLECTION_NAME,
    "candidate_k_default": 50,
    "top_k_default": 5,
    "ranking_weights": RANKING_WEIGHTS,
    "ingredient_aliases": INGREDIENT_ALIASES,
    "concern_ingredient_weights": CONCERN_INGREDIENT_WEIGHTS,
    "catalog_rows": int(len(catalog)),
    "catalog_columns": int(len(catalog.columns)),
    "ingredient_profiles_available": int(sum(p["available"] for p in ingredient_profiles.values())),
    "ingredient_profiles_unavailable": int(sum(not p["available"] for p in ingredient_profiles.values())),
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}

ENGINE_CONFIG_PATH.write_text(
    json.dumps(ENGINE_CONFIG, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with INGREDIENT_PROFILES_PATH.open("w", encoding="utf-8") as handle:
    for pid, profile in ingredient_profiles.items():
        record = {"product_id": pid, **profile}
        handle.write(
            json.dumps(record, ensure_ascii=False) + chr(10)
        )

application_columns = [
    c
    for c in [
        "product_id", "product_name", "brand_name",
        "primary_category", "secondary_category", "tertiary_category",
        "effective_price_usd", "rating", "reviews",
        "review_count_observed", "recommendation_rate", "has_reviews",
        "ingredients", "ingredients_clean", "ingredient_tokens",
        "ingredient_count", "highlights", "product_document",
    ]
    if c in catalog.columns
]

catalog[application_columns].to_csv(
    APPLICATION_CATALOG_PATH,
    index=False,
)

print("Saved artifacts:")
print("  Embeddings:", EMBEDDINGS_PATH)
print("  Embedding IDs:", EMBEDDING_IDS_PATH)
print("  ChromaDB:", CHROMA_PATH)
print("  Engine config:", ENGINE_CONFIG_PATH)
print("  Ingredient profiles:", INGREDIENT_PROFILES_PATH)
print("  Application catalog:", APPLICATION_CATALOG_PATH)

for required_path in [
    EMBEDDINGS_PATH, EMBEDDING_IDS_PATH, ENGINE_CONFIG_PATH,
    INGREDIENT_PROFILES_PATH, APPLICATION_CATALOG_PATH,
]:
    assert required_path.exists(), f"Missing artifact: {required_path}"

print("Artifact saving: PASS")

In [36]:
# ============================================================
# 25. Save Application Artifacts
# ============================================================

print("=" * 100)
print("SAVING APPLICATION ARTIFACTS")
print("=" * 100)

# ------------------------------------------------------------
# 1. Create the artifact directory
# ------------------------------------------------------------

ARTIFACT_DIR = Path(PROJECT_ROOT) / "data" / "artifacts"
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Artifact directory:")
print(ARTIFACT_DIR)
print()


# ------------------------------------------------------------
# 2. Define ALL artifact paths explicitly
# ------------------------------------------------------------

EMBEDDINGS_PATH = (
    ARTIFACT_DIR
    / "product_embeddings.npy"
)

EMBEDDING_IDS_PATH = (
    ARTIFACT_DIR
    / "product_embedding_ids.json"
)

CHROMA_PATH = (
    ARTIFACT_DIR
    / "chroma"
)

ENGINE_CONFIG_PATH = (
    ARTIFACT_DIR
    / "recommendation_engine_config.json"
)

INGREDIENT_PROFILES_PATH = (
    ARTIFACT_DIR
    / "ingredient_profiles.jsonl"
)

APPLICATION_CATALOG_PATH = (
    ARTIFACT_DIR
    / "application_catalog.csv"
)


# ------------------------------------------------------------
# 3. Save embeddings
# ------------------------------------------------------------

np.save(
    EMBEDDINGS_PATH,
    np.asarray(
        product_embeddings,
        dtype=np.float32,
    ),
)

EMBEDDING_IDS_PATH.write_text(
    json.dumps(
        product_ids,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 4. Save canonical ingredient profiles
# ------------------------------------------------------------

with INGREDIENT_PROFILES_PATH.open(
    "w",
    encoding="utf-8",
) as handle:

    for product_id, profile in ingredient_profiles.items():

        record = {
            "product_id": str(product_id),
            **profile,
        }

        handle.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


# ------------------------------------------------------------
# 5. Save recommendation-engine configuration
# ------------------------------------------------------------

ENGINE_CONFIG = {
    "project": "DermaMatch AI",

    "embedding_model": (
        EMBEDDING_MODEL_NAME
    ),

    "embedding_dimension": int(
        EMBEDDING_DIM
    ),

    "chroma_collection": (
        CHROMA_COLLECTION_NAME
    ),

    "chroma_path": str(
        CHROMA_PATH
    ),

    "candidate_k_default": 50,

    "top_k_default": 5,

    "ranking_weights": {
        key: float(value)
        for key, value
        in RANKING_WEIGHTS.items()
    },

    "ingredient_aliases": (
        INGREDIENT_ALIASES
    ),

    "concern_ingredient_weights": (
        CONCERN_INGREDIENT_WEIGHTS
    ),

    "catalog_rows": int(
        len(catalog)
    ),

    "catalog_columns": int(
        len(catalog.columns)
    ),

    "ingredient_profiles_available": int(
        sum(
            bool(
                profile["available"]
            )
            for profile
            in ingredient_profiles.values()
        )
    ),

    "ingredient_profiles_unavailable": int(
        sum(
            not bool(
                profile["available"]
            )
            for profile
            in ingredient_profiles.values()
        )
    ),

    "generated_at_utc": (
        pd.Timestamp.now(
            tz="UTC"
        ).isoformat()
    ),
}


ENGINE_CONFIG_PATH.write_text(
    json.dumps(
        ENGINE_CONFIG,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 6. Save application catalog
# ------------------------------------------------------------

APPLICATION_COLUMNS = [
    column
    for column in [
        "product_id",
        "product_name",
        "brand_name",
        "primary_category",
        "secondary_category",
        "tertiary_category",
        "effective_price_usd",
        "rating",
        "reviews",
        "review_count_observed",
        "recommendation_rate",
        "has_reviews",
        "ingredients",
        "ingredients_clean",
        "ingredient_tokens",
        "ingredient_count",
        "highlights",
        "product_document",
    ]
    if column in catalog.columns
]

catalog[
    APPLICATION_COLUMNS
].to_csv(
    APPLICATION_CATALOG_PATH,
    index=False,
)


# ------------------------------------------------------------
# 7. Verify ChromaDB path
# ------------------------------------------------------------

CHROMA_PATH.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 8. Print saved artifacts
# ------------------------------------------------------------

print("Saved artifacts")
print("-" * 100)

print(
    "Embeddings:",
    EMBEDDINGS_PATH,
)

print(
    "Embedding IDs:",
    EMBEDDING_IDS_PATH,
)

print(
    "ChromaDB:",
    CHROMA_PATH,
)

print(
    "Engine config:",
    ENGINE_CONFIG_PATH,
)

print(
    "Ingredient profiles:",
    INGREDIENT_PROFILES_PATH,
)

print(
    "Application catalog:",
    APPLICATION_CATALOG_PATH,
)


# ------------------------------------------------------------
# 9. Final artifact validation
# ------------------------------------------------------------

required_artifacts = [
    EMBEDDINGS_PATH,
    EMBEDDING_IDS_PATH,
    ENGINE_CONFIG_PATH,
    INGREDIENT_PROFILES_PATH,
    APPLICATION_CATALOG_PATH,
]

print("\nArtifact validation")
print("-" * 100)

for artifact in required_artifacts:

    exists = artifact.exists()

    print(
        f"{artifact.name}:",
        "PASS" if exists else "FAIL",
    )

    assert exists, (
        f"Required artifact was not created: {artifact}"
    )


# Additional content checks
assert (
    EMBEDDINGS_PATH.stat().st_size > 0
)

assert (
    EMBEDDING_IDS_PATH.stat().st_size > 0
)

assert (
    ENGINE_CONFIG_PATH.stat().st_size > 0
)

assert (
    INGREDIENT_PROFILES_PATH.stat().st_size > 0
)

assert (
    APPLICATION_CATALOG_PATH.stat().st_size > 0
)

print("\n[PASS] All application artifacts saved successfully.")

SAVING APPLICATION ARTIFACTS
Artifact directory:
D:\CODE\ORBO.ai\data\artifacts

Saved artifacts
----------------------------------------------------------------------------------------------------
Embeddings: D:\CODE\ORBO.ai\data\artifacts\product_embeddings.npy
Embedding IDs: D:\CODE\ORBO.ai\data\artifacts\product_embedding_ids.json
ChromaDB: D:\CODE\ORBO.ai\data\artifacts\chroma
Engine config: D:\CODE\ORBO.ai\data\artifacts\recommendation_engine_config.json
Ingredient profiles: D:\CODE\ORBO.ai\data\artifacts\ingredient_profiles.jsonl
Application catalog: D:\CODE\ORBO.ai\data\artifacts\application_catalog.csv

Artifact validation
----------------------------------------------------------------------------------------------------
product_embeddings.npy: PASS
product_embedding_ids.json: PASS
recommendation_engine_config.json: PASS
ingredient_profiles.jsonl: PASS
application_catalog.csv: PASS

[PASS] All application artifacts saved successfully.


# 27. Handoff to Flask + Streamlit

The recommendation engine is now a self-contained backend component.

## Production flow

```text
Streamlit UI
      ↓
Flask REST API
      ↓
recommend(...)
      ↓
ChromaDB + scoring + ranking
      ↓
Top-K products + deterministic explanations
```

### Core application artifacts

```text
data/artifacts/
├── product_embeddings.npy
├── product_embedding_ids.json
├── ingredient_profiles.jsonl
├── recommendation_engine_config.json
├── application_catalog.csv
└── chroma/
```

### Engineering rule

The Flask API should **import and call `recommend()`**. It should not duplicate the ingredient matcher, scoring formulas, or ranking logic.

The Streamlit UI should call Flask rather than implement a second recommendation engine.

### Final project status

**Part 1:** Data pipeline — complete  
**Part 2:** Recommendation engine — complete after this validation passes  
**Part 3:** Flask + Streamlit — next